# Kaggriculture v25 — New Production Routes with Reactive Sales

This notebook contains the complete agent source and builds a ready-to-submit
`submission_competitive_v25.tar.gz`. No dataset, attached `main.py`, donor download,
KaggleHub call, or private input is required. Run the four code cells in order.

The production base is Yusuke Hayashi's public **Shop Router 0908** portfolio:
four action schedules, an early route choice from revealed yarn shops, and a late
branch from the observed egg market inventory. This is a new production base,
not an incremental v24 market-order change.

The runtime aligns worker actions, repairs obstructing weeds, advances feasible
finished-product sales by one turn when no town consumption intervenes, and
suppresses the corresponding quantities on the following turn. On the final
action it drops cargo already within shed access and liquidates available products.
Only the current public observation and our own private inventory are used.

The agent itself uses only Python's standard library. The notebook installs the
official engine only if the required version is absent. Internet is needed for
that installation only; agent execution and archive creation are offline.

Local results do not imply a guaranteed live ladder rating. The archive has not
been submitted automatically. See the accompanying EXP-149 report for complete
paired results and limitations.

## Local confirmation

| Agent | W/L/T | Win rate | Mean margin |
|---|---:|---:|---:|
| v23 | 514/110/0 | 82.37% | +24,404.26 |
| v24 | 529/95/0 | 84.78% | +24,467.64 |
| v25 | 582/42/0 | 93.27% | +8,254.51 |

Official engine 1.32.7; 13 public policies, 24 held-out seeds, both seats (624 games per agent).


## 1. Prepare the official engine


In [ ]:
from pathlib import Path
import importlib.metadata, os, subprocess, sys

WORKDIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
try:
    engine_version = importlib.metadata.version('kaggle-environments')
except importlib.metadata.PackageNotFoundError:
    engine_version = None
if engine_version != '1.32.7':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'kaggle-environments==1.32.7'], check=True)
print('Working directory:', WORKDIR)
print('Engine:', importlib.metadata.version('kaggle-environments'))


## 2. Write the complete embedded agent

Run this cell unchanged. There is no input-file search.


In [ ]:
%%writefile main.py
# Kaggriculture v25, EXP-149: Shop0908 production, sale lead, terminal cargo rescue.
# Runtime chassis: Apache-2.0; thomastschinkel, yhay81, tetsutani.
# Routing and public action data: yhay81/shop-router-0908, frozen September 8, 2026.
# 
#                                  Apache License
#                            Version 2.0, January 2004
#                         http://www.apache.org/licenses/
# 
#    TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION
# 
#    1. Definitions.
# 
#       "License" shall mean the terms and conditions for use, reproduction,
#       and distribution as defined by Sections 1 through 9 of this document.
# 
#       "Licensor" shall mean the copyright owner or entity authorized by
#       the copyright owner that is granting the License.
# 
#       "Legal Entity" shall mean the union of the acting entity and all
#       other entities that control, are controlled by, or are under common
#       control with that entity. For the purposes of this definition,
#       "control" means (i) the power, direct or indirect, to cause the
#       direction or management of such entity, whether by contract or
#       otherwise, or (ii) ownership of fifty percent (50%) or more of the
#       outstanding shares, or (iii) beneficial ownership of such entity.
# 
#       "You" (or "Your") shall mean an individual or Legal Entity
#       exercising permissions granted by this License.
# 
#       "Source" form shall mean the preferred form for making modifications,
#       including but not limited to software source code, documentation
#       source, and configuration files.
# 
#       "Object" form shall mean any form resulting from mechanical
#       transformation or translation of a Source form, including but
#       not limited to compiled object code, generated documentation,
#       and conversions to other media types.
# 
#       "Work" shall mean the work of authorship, whether in Source or
#       Object form, made available under the License, as indicated by a
#       copyright notice that is included in or attached to the work
#       (an example is provided in the Appendix below).
# 
#       "Derivative Works" shall mean any work, whether in Source or Object
#       form, that is based on (or derived from) the Work and for which the
#       editorial revisions, annotations, elaborations, or other modifications
#       represent, as a whole, an original work of authorship. For the purposes
#       of this License, Derivative Works shall not include works that remain
#       separable from, or merely link (or bind by name) to the interfaces of,
#       the Work and Derivative Works thereof.
# 
#       "Contribution" shall mean any work of authorship, including
#       the original version of the Work and any modifications or additions
#       to that Work or Derivative Works thereof, that is intentionally
#       submitted to Licensor for inclusion in the Work by the copyright owner
#       or by an individual or Legal Entity authorized to submit on behalf of
#       the copyright owner. For the purposes of this definition, "submitted"
#       means any form of electronic, verbal, or written communication sent
#       to the Licensor or its representatives, including but not limited to
#       communication on electronic mailing lists, source code control systems,
#       and issue tracking systems that are managed by, or on behalf of, the
#       Licensor for the purpose of discussing and improving the Work, but
#       excluding communication that is conspicuously marked or otherwise
#       designated in writing by the copyright owner as "Not a Contribution."
# 
#       "Contributor" shall mean Licensor and any individual or Legal Entity
#       on behalf of whom a Contribution has been received by Licensor and
#       subsequently incorporated within the Work.
# 
#    2. Grant of Copyright License. Subject to the terms and conditions of
#       this License, each Contributor hereby grants to You a perpetual,
#       worldwide, non-exclusive, no-charge, royalty-free, irrevocable
#       copyright license to reproduce, prepare Derivative Works of,
#       publicly display, publicly perform, sublicense, and distribute the
#       Work and such Derivative Works in Source or Object form.
# 
#    3. Grant of Patent License. Subject to the terms and conditions of
#       this License, each Contributor hereby grants to You a perpetual,
#       worldwide, non-exclusive, no-charge, royalty-free, irrevocable
#       (except as stated in this section) patent license to make, have made,
#       use, offer to sell, sell, import, and otherwise transfer the Work,
#       where such license applies only to those patent claims licensable
#       by such Contributor that are necessarily infringed by their
#       Contribution(s) alone or by combination of their Contribution(s)
#       with the Work to which such Contribution(s) was submitted. If You
#       institute patent litigation against any entity (including a
#       cross-claim or counterclaim in a lawsuit) alleging that the Work
#       or a Contribution incorporated within the Work constitutes direct
#       or contributory patent infringement, then any patent licenses
#       granted to You under this License for that Work shall terminate
#       as of the date such litigation is filed.
# 
#    4. Redistribution. You may reproduce and distribute copies of the
#       Work or Derivative Works thereof in any medium, with or without
#       modifications, and in Source or Object form, provided that You
#       meet the following conditions:
# 
#       (a) You must give any other recipients of the Work or
#           Derivative Works a copy of this License; and
# 
#       (b) You must cause any modified files to carry prominent notices
#           stating that You changed the files; and
# 
#       (c) You must retain, in the Source form of any Derivative Works
#           that You distribute, all copyright, patent, trademark, and
#           attribution notices from the Source form of the Work,
#           excluding those notices that do not pertain to any part of
#           the Derivative Works; and
# 
#       (d) If the Work includes a "NOTICE" text file as part of its
#           distribution, then any Derivative Works that You distribute must
#           include a readable copy of the attribution notices contained
#           within such NOTICE file, excluding those notices that do not
#           pertain to any part of the Derivative Works, in at least one
#           of the following places: within a NOTICE text file distributed
#           as part of the Derivative Works; within the Source form or
#           documentation, if provided along with the Derivative Works; or,
#           within a display generated by the Derivative Works, if and
#           wherever such third-party notices normally appear. The contents
#           of the NOTICE file are for informational purposes only and
#           do not modify the License. You may add Your own attribution
#           notices within Derivative Works that You distribute, alongside
#           or as an addendum to the NOTICE text from the Work, provided
#           that such additional attribution notices cannot be construed
#           as modifying the License.
# 
#       You may add Your own copyright statement to Your modifications and
#       may provide additional or different license terms and conditions
#       for use, reproduction, or distribution of Your modifications, or
#       for any such Derivative Works as a whole, provided Your use,
#       reproduction, and distribution of the Work otherwise complies with
#       the conditions stated in this License.
# 
#    5. Submission of Contributions. Unless You explicitly state otherwise,
#       any Contribution intentionally submitted for inclusion in the Work
#       by You to the Licensor shall be under the terms and conditions of
#       this License, without any additional terms or conditions.
#       Notwithstanding the above, nothing herein shall supersede or modify
#       the terms of any separate license agreement you may have executed
#       with Licensor regarding such Contributions.
# 
#    6. Trademarks. This License does not grant permission to use the trade
#       names, trademarks, service marks, or product names of the Licensor,
#       except as required for reasonable and customary use in describing the
#       origin of the Work and reproducing the content of the NOTICE file.
# 
#    7. Disclaimer of Warranty. Unless required by applicable law or
#       agreed to in writing, Licensor provides the Work (and each
#       Contributor provides its Contributions) on an "AS IS" BASIS,
#       WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
#       implied, including, without limitation, any warranties or conditions
#       of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
#       PARTICULAR PURPOSE. You are solely responsible for determining the
#       appropriateness of using or redistributing the Work and assume any
#       risks associated with Your exercise of permissions under this License.
# 
#    8. Limitation of Liability. In no event and under no legal theory,
#       whether in tort (including negligence), contract, or otherwise,
#       unless required by applicable law (such as deliberate and grossly
#       negligent acts) or agreed to in writing, shall any Contributor be
#       liable to You for damages, including any direct, indirect, special,
#       incidental, or consequential damages of any character arising as a
#       result of this License or out of the use or inability to use the
#       Work (including but not limited to damages for loss of goodwill,
#       work stoppage, computer failure or malfunction, or any and all
#       other commercial damages or losses), even if such Contributor
#       has been advised of the possibility of such damages.
# 
#    9. Accepting Warranty or Additional Liability. While redistributing
#       the Work or Derivative Works thereof, You may choose to offer,
#       and charge a fee for, acceptance of support, warranty, indemnity,
#       or other liability obligations and/or rights consistent with this
#       License. However, in accepting such obligations, You may act only
#       on Your own behalf and on Your sole responsibility, not on behalf
#       of any other Contributor, and only if You agree to indemnify,
#       defend, and hold each Contributor harmless for any liability
#       incurred by, or claims asserted against, such Contributor by reason
#       of your accepting any such warranty or additional liability.
# 
#    END OF TERMS AND CONDITIONS
# 
#    APPENDIX: How to apply the Apache License to your work.
# 
#       To apply the Apache License to your work, attach the following
#       boilerplate notice, with the fields enclosed by brackets "[]"
#       replaced with your own identifying information. (Don't include
#       the brackets!)  The text should be enclosed in the appropriate
#       comment syntax for the file format. We also recommend that a
#       file or class name and description of purpose be included on the
#       same "printed page" as the copyright notice for easier
#       identification within third-party archives.
# 
#    Copyright [yyyy] [name of copyright owner]
# 
#    Licensed under the Apache License, Version 2.0 (the "License");
#    you may not use this file except in compliance with the License.
#    You may obtain a copy of the License at
# 
#        http://www.apache.org/licenses/LICENSE-2.0
# 
#    Unless required by applicable law or agreed to in writing, software
#    distributed under the License is distributed on an "AS IS" BASIS,
#    WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#    See the License for the specific language governing permissions and
#    limitations under the License.
"""Kaggriculture route-replay chassis (pure Python, stdlib only).

A *route* is a pre-computed tape of 719 Kaggle-format actions
``{"farmer": [op, ...], "hands": [[op, ...], ...], "market": [[order, item, qty], ...]}``.
The chassis replays the tape chosen by a caller-supplied ``router`` and wraps it
in small reactive layers (each independently switchable via ``settings``):

    hand_align            pad/truncate hands to the real hand count      (fieldbook_logic)
    weed_repair           DIG a weed that blocks PLANT/BUILD, replay      (tetsutani + task spec)
    sell_lead             sell next step's lots one step early            (fieldbook _lead_sale)
    front_run             sell before the opponent's scheduled SELL       (hook; opponent_plan)
    budget_guard          fund each 72-step block's purchases             (six_day_budget_guard.hpp)
    room_guard            keep shed <= 99 at hour 23                      (tetsutani)
    clamp_sells           trim SELL orders to the projected shed          (tetsutani)
    dead_stock            sell stock the route will never sell            (tetsutani)
    terminal_liquidation  step >= 718: sell the whole projected shed      (fieldbook _terminal_sale)

Engine facts (verified against kaggle_environments 1.32.7, env_1_32_7.py):
  observation["farms"][p] = {"money", "tiles"[y][x], "farmer"[x,y], "hands"[[x,y]..],
                             "unlocked_quadrants", "hires_today"}
  tiles: None (empty) | "LOCKED" | {"kind": WEED|COOP|PASTURE|PLANT, "crop"/"animal", ...}
  observation["private"] = {"shed": {item: n}, "seeds": {crop: n}, "inventories": [{}...]}
  observation["market"] = {"inventory": {...}, "prices": {...}}
  observation["town"] = {"unlocked_shops": [...]}
  Agents act on steps 0..718 (interpreter marks DONE once step >= episodeSteps-2).
"""
from __future__ import annotations

import copy

PRODUCTS = ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER")
SEED_PRICE = {"WHEAT": 10, "CARROT": 20, "TOMATO": 50, "STRAWBERRY": 100, "MELON": 80}
ANIMAL_COST = {"GOOSE": 300, "COW": 400, "SHEEP": 500}
ANIMAL_STRUCTURE = {"GOOSE": "COOP", "COW": "PASTURE", "SHEEP": "PASTURE"}
LAND_PRICES = (1000, 2000, 4000)
MOVES = {"NORTH": (0, -1), "SOUTH": (0, 1), "EAST": (1, 0), "WEST": (-1, 0)}
FRONT_RUN_ITEMS = ("MILK", "WOOL", "STRAWBERRY", "MELON")
LAST_ACT_STEP = 718
PASS_ACTION = {"farmer": ["PASS"], "hands": [], "market": []}

DEFAULT_SETTINGS = {
    "hand_align": True,
    "weed_repair": True,
    "sell_lead": True,
    "front_run": True,
    "budget_guard": True,
    "room_guard": True,
    "clamp_sells": True,
    "dead_stock": True,
    "terminal_liquidation": True,
    # tunables
    "block_turns": 72,
    "shed_capacity": 100,
    "board_size": 10,
    "max_orders": 10,
    "turns_per_day": 24,
    "min_sell_price": 2,
}


# --------------------------------------------------------------------------- helpers
def _get(value, key, default=None):
    """Field access that works for dicts and Kaggle Struct/attribute objects."""
    if isinstance(value, dict):
        return value.get(key, default)
    getter = getattr(value, "get", None)
    if callable(getter):
        return getter(key, default)
    return getattr(value, key, default)


def _int(value, default=0):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def _fib(n):
    a, b = 1, 1
    for _ in range(n):
        a, b = b, a + b
    return a


def _step_of(observation):
    raw = _get(observation, "step")
    if raw is not None:
        return _int(raw)
    return _int(_get(observation, "day", 0)) * 24 + _int(_get(observation, "hour", 0))


def _shed_adjacent(pos, board):
    if not isinstance(pos, (list, tuple)) or len(pos) < 2:
        return False
    half = board // 2
    return pos[0] in (half - 1, half) and pos[1] in (half - 1, half)


def _tile_at(tiles, pos):
    try:
        x, y = int(pos[0]), int(pos[1])
        return tiles[y][x]
    except (TypeError, ValueError, IndexError):
        return "LOCKED"


def _is_noop(act, tile, inv, seeds, pos, board):
    """True when the engine will certainly ignore ``act`` (mirrors _apply_unit_action)."""
    if not act:
        return True
    op = act[0]
    x, y = pos[0], pos[1]
    if op in MOVES:
        dx, dy = MOVES[op]
        return not (0 <= x + dx < board and 0 <= y + dy < board)
    if op == "PASS":
        return True
    adjacent = _shed_adjacent(pos, board)
    if op == "DROP":
        return (not adjacent) or (not inv)
    if op == "PICKUP":
        return not adjacent
    if op == "PLACE":
        item = act[1] if len(act) > 1 else None
        if item in ANIMAL_STRUCTURE and isinstance(tile, dict) \
                and _get(tile, "kind") == ANIMAL_STRUCTURE[item] and _get(tile, "animal") is None:
            return _int(_get(inv, item, 0)) <= 0
        return (not adjacent) or _int(_get(inv, item, 0)) <= 0
    if tile == "LOCKED":
        return True
    is_dict = isinstance(tile, dict)
    kind = _get(tile, "kind") if is_dict else None
    animal = is_dict and _get(tile, "animal") is not None
    if op == "PLANT":
        return tile is not None or _int(_get(seeds, act[1] if len(act) > 1 else None, 0)) <= 0
    if op == "WATER":
        return kind != "PLANT" or bool(_get(tile, "watered_today"))
    if op == "HARVEST":
        return (not is_dict) or _int(_get(tile, "yield_units", 0)) <= 0
    if op == "FERTILIZE":
        return kind != "PLANT" or _int(_get(inv, "FERTILIZER", 0)) <= 0
    if op == "DIG":
        return tile is None or animal
    if op in ("BUILD_COOP", "BUILD_PASTURE"):
        return tile is not None
    if op == "FEED":
        return (not animal) or bool(_get(tile, "fed_today")) or _int(_get(inv, "WHEAT", 0)) <= 0
    if op == "COLLECT_FERTILIZER":
        return (not animal) or (not _get(tile, "fertilizer_available"))
    if op == "CARE":
        return (not animal) or bool(_get(tile, "cared_today"))
    return True


class _View:
    """Cheap per-step snapshot of everything the layers read from the observation."""

    def __init__(self, observation, player, cfg):
        farms = list(_get(observation, "farms", []) or [])
        self.farm = farms[player] if player < len(farms) else {}
        self.rival = farms[1 - player] if len(farms) >= 2 and 1 - player < len(farms) else {}
        private = _get(observation, "private", {}) or {}
        self.shed = {k: max(0, _int(v)) for k, v in dict(_get(private, "shed", {}) or {}).items()}
        self.seeds = dict(_get(private, "seeds", {}) or {})
        self.invs = [dict(i or {}) for i in (_get(private, "inventories", []) or [])]
        market = _get(observation, "market", {}) or {}
        self.prices = {k: _int(v) for k, v in dict(_get(market, "prices", {}) or {}).items()}
        self.money = float(_get(self.farm, "money", 0.0) or 0.0)
        self.tiles = _get(self.farm, "tiles", []) or []
        self.board = len(self.tiles) or cfg["board_size"]
        self.positions = [_get(self.farm, "farmer", None)] + [list(p) for p in (_get(self.farm, "hands", []) or [])]
        self.hires_today = _int(_get(self.farm, "hires_today", 0))
        self.quadrants = len(list(_get(self.farm, "unlocked_quadrants", []) or []))

    def inv(self, idx):
        return self.invs[idx] if idx < len(self.invs) else {}

    def in_hands(self, item):
        return sum(max(0, _int(_get(inv, item, 0))) for inv in self.invs)


# --------------------------------------------------------------------------- chassis
class Chassis:
    """Replays ``routes[router(...)]`` with reactive safety/market layers.

    routes         : {route_id: list of >= 719 Kaggle action dicts}
    router         : callable(observation, step, state_dict) -> route_id, called every
                     step; ``state_dict`` is per-player and persists across the game.
    settings       : overrides for DEFAULT_SETTINGS (layer switches + tunables)
    opponent_plan  : optional list of the opponent's expected actions (front_run hook)
    """

    def __init__(self, routes, router=None, settings=None, opponent_plan=None):
        self.routes = {rid: list(tape) for rid, tape in routes.items()}
        self.router = router or (lambda observation, step, state: next(iter(self.routes)))
        self.cfg = dict(DEFAULT_SETTINGS)
        self.cfg.update(settings or {})
        self.opponent_plan = opponent_plan
        self.players = {}
        self.diagnostics = {"layer_fallbacks": 0, "entry_fallbacks": 0}
        self._future_sells = {}   # route id -> {item: [remaining planned SELL qty from step t]}

    # ---- state -----------------------------------------------------------------
    def _state(self, player, step):
        st = self.players.get(player)
        if st is None or step == 0 or step <= st["last_step"]:
            st = {"last_step": -1, "route": None, "router_state": {},
                  "pending": {}, "sell_state": {"due_step": -1, "suppress": {}}}
            self.players[player] = st
        st["last_step"] = step
        return st

    def _route_action(self, route, step):
        tape = self.routes[route]
        if 0 <= step < len(tape) and isinstance(tape[step], dict):
            return copy.deepcopy(tape[step])
        return copy.deepcopy(PASS_ACTION)

    def future_sells(self, route, item, step):
        """Planned SELL quantity of ``item`` in route steps >= ``step`` (suffix sums)."""
        table = self._future_sells.get(route)
        if table is None:
            tape = self.routes[route]
            n = len(tape)
            table = {p: [0] * (n + 1) for p in PRODUCTS}
            for t in range(n - 1, -1, -1):
                for p in PRODUCTS:
                    table[p][t] = table[p][t + 1]
                for o in (tape[t].get("market") or []) if isinstance(tape[t], dict) else []:
                    if o and o[0] == "SELL" and len(o) >= 3 and o[1] in table:
                        table[o[1]][t] += max(0, _int(o[2]))
            self._future_sells[route] = table
        col = table.get(item)
        return col[step] if col and 0 <= step < len(col) else 0

    # ---- main entry -----------------------------------------------------------
    def act(self, observation, configuration=None):
        if len(_get(observation, "farms", []) or []) < 2:
            raise ValueError("incomplete observation")  # factory falls back to tape
        step = _step_of(observation)
        player = _int(_get(observation, "player", 0))
        st = self._state(player, step)
        cfg = self.cfg
        view = _View(observation, player, cfg)

        route = self.router(observation, step, st["router_state"])
        if route not in self.routes:
            route = st["route"] if st["route"] in self.routes else next(iter(self.routes))
        st["route"] = route
        action = self._route_action(route, step)
        raw = copy.deepcopy(action)
        try:
            if cfg["hand_align"]:
                self._hand_align(action, view)
            if cfg["weed_repair"]:
                self._weed_repair(action, view, st, route, step)
            if cfg["sell_lead"] or cfg["front_run"]:
                self._apply_suppression(action, st["sell_state"], step)
            projected = self._projected_shed(action, view)
            lead_available = dict(projected)
            next_sup = {"due_step": -1, "suppress": {}}
            if cfg["sell_lead"]:
                self._sell_lead(action, view, lead_available, route, step, next_sup)
            if cfg["front_run"] and self.opponent_plan:
                self._front_run(action, view, lead_available, route, step, next_sup)
            st["sell_state"] = next_sup
            if cfg["budget_guard"]:
                self._budget_guard(action, view, route, step)
            if cfg["room_guard"]:
                self._room_guard(action, view, route, step)
            if cfg["clamp_sells"]:
                self._clamp_sells(action, projected)
            if cfg["dead_stock"]:
                self._dead_stock(action, view, projected, route, step)
            if cfg["terminal_liquidation"]:
                self._terminal_liquidation(action, projected, step)
            action["market"] = action["market"][: cfg["max_orders"]]
            return action
        except Exception:
            self.diagnostics["layer_fallbacks"] += 1
            return raw

    # ---- layer: hand_align ----------------------------------------------------
    def _hand_align(self, action, view):
        """Pad with PASS / truncate the tape's hand list to the real number of hands
        (fieldbook_logic.act). Extra hands would be ignored by the engine anyway;
        missing ones just idle, so alignment only tidies the action."""
        expected = max(0, len(view.positions) - 1)
        hands = list(action.get("hands") or [])
        hands.extend([["PASS"] for _ in range(max(0, expected - len(hands)))])
        action["hands"] = hands[:expected]

    # ---- layer: weed_repair ---------------------------------------------------
    def _weed_repair(self, action, view, st, route, step):
        """If a PLANT/BUILD_* target tile is a WEED, DIG now and queue the intended
        action for that unit; the queue replays on a later step when the unit still
        stands there and its tape action would be a no-op (the displaced no-op is
        queued behind it, so PLANT -> WATER chains survive). A PLANT is only replayed
        when the unit's next tape action is not a move, so the mandatory same-day
        WATER can follow; otherwise the seed is kept. A no-op turn spent on a weed
        is also converted to DIG (tetsutani weed_dig)."""
        units = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])
        pending = st["pending"]
        tape = self.routes[route]
        nxt = tape[step + 1] if step + 1 < len(tape) and isinstance(tape[step + 1], dict) else {}
        next_units = [nxt.get("farmer") or ["PASS"]] + list(nxt.get("hands") or [])
        for i in range(min(len(units), len(view.positions))):
            pos = view.positions[i]
            if not isinstance(pos, (list, tuple)):
                continue
            pos = (int(pos[0]), int(pos[1]))
            tile = _tile_at(view.tiles, pos)
            act = list(units[i])
            queue = pending.get(i)
            if queue and queue[0][0] != pos:
                pending.pop(i, None)
                queue = None
            is_weed = isinstance(tile, dict) and _get(tile, "kind") == "WEED"
            noop = _is_noop(act, tile, view.inv(i), view.seeds, pos, view.board)
            next_op = next_units[i][0] if i < len(next_units) and next_units[i] else "PASS"
            if act and act[0] in ("PLANT", "BUILD_COOP", "BUILD_PASTURE") and is_weed:
                pending.setdefault(i, []).append((pos, act))
                act = ["DIG"]
            elif queue and noop:
                _, replay = queue[0]
                if replay[0] == "PLANT" and next_op in MOVES:
                    pending.pop(i, None)          # WATER could never follow: keep the seed
                else:
                    queue.pop(0)
                    if act and act[0] != "PASS" and act[0] not in MOVES:
                        queue.append((pos, act))
                    act = replay
                    if not queue:
                        pending.pop(i, None)
            elif is_weed and noop:
                act = ["DIG"]
            units[i] = act
        action["farmer"] = units[0]
        action["hands"] = units[1:]

    # ---- projected shed -------------------------------------------------------
    def _projected_shed(self, action, view):
        """Shed contents after this step's unit actions but before the market runs:
        PICKUP removes, DROP/PLACE(non-animal) near the shed adds up to capacity
        (fieldbook _projected_shed / tetsutani projected shed)."""
        cap = self.cfg["shed_capacity"]
        proj = {p: view.shed.get(p, 0) for p in PRODUCTS}
        for k, v in view.shed.items():
            proj.setdefault(k, v)
        total = sum(proj.values())
        units = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])
        for i in range(min(len(units), len(view.positions))):
            if not _shed_adjacent(view.positions[i], view.board):
                continue
            act = units[i]
            op = act[0] if act else "PASS"
            inv = view.inv(i)
            if op == "PICKUP" and len(act) >= 2 and act[1] in proj:
                qty = min(proj[act[1]], max(0, _int(act[2]) if len(act) >= 3 else 1))
                proj[act[1]] -= qty
                total -= qty
            elif op == "DROP":
                for item, held in inv.items():
                    take = min(max(0, _int(held)), max(0, cap - total))
                    if take > 0:
                        proj[item] = proj.get(item, 0) + take
                        total += take
            elif op == "PLACE" and len(act) >= 2 and act[1] not in ANIMAL_STRUCTURE:
                item = act[1]
                take = min(max(0, _int(act[2]) if len(act) >= 3 else 1),
                           max(0, _int(_get(inv, item, 0))), max(0, cap - total))
                if take > 0:
                    proj[item] = proj.get(item, 0) + take
                    total += take
        return proj

    # ---- layer: sell_lead / front_run suppression ------------------------------
    @staticmethod
    def _apply_suppression(action, sell_state, step):
        """Remove from this step's SELLs the quantities already sold a step early."""
        if sell_state.get("due_step") != step:
            return
        remaining = dict(sell_state.get("suppress", {}))
        kept = []
        for order in action.get("market") or []:
            order = list(order)
            if order and order[0] == "SELL" and len(order) >= 3 and remaining.get(order[1], 0) > 0:
                removed = min(max(0, _int(order[2])), remaining[order[1]])
                order[2] = _int(order[2]) - removed
                remaining[order[1]] -= removed
                # A zero-quantity order keeps later market race slots intact.
            kept.append(order)
        action["market"] = kept

    @staticmethod
    def _add_sell(action, item, qty, max_orders, merge=True):
        market = action.setdefault("market", [])
        if merge:
            for order in market:
                if order and order[0] == "SELL" and order[1] == item:
                    order[2] = _int(order[2]) + qty
                    return True
        if len(market) >= max_orders:
            return False
        market.append(["SELL", item, qty])
        return True

    def _sell_lead(self, action, view, projected, route, step, next_sup):
        """fieldbook _lead_sale: when step % 4 != 0 (no town consumption between the
        two steps) sell the lots the tape plans to SELL next step now, for products
        other than WHEAT/FERTILIZER we already hold, and suppress them next step.
        Skipped at the last step, at shop-unlock boundaries and if a SELL for that
        product is already queued this step."""
        cfg = self.cfg
        nxt = step + 1
        unlock_period = 3 * cfg["turns_per_day"]
        if nxt > LAST_ACT_STEP or nxt % unlock_period == 0 or step % 4 == 0:
            return
        tape = self.routes[route]
        future = tape[nxt] if nxt < len(tape) and isinstance(tape[nxt], dict) else {}
        planned = {}
        for o in future.get("market") or []:
            if o and o[0] == "SELL" and len(o) >= 3 and o[1] in PRODUCTS:
                planned[o[1]] = planned.get(o[1], 0) + max(0, _int(o[2]))
        already = {o[1] for o in action.get("market") or [] if o and o[0] == "SELL" and len(o) > 1}
        for item in PRODUCTS:
            if item in ("WHEAT", "FERTILIZER") or planned.get(item, 0) <= 0 or item in already:
                continue
            qty = min(projected.get(item, 0), planned[item])
            if qty <= 0 or view.prices.get(item, 0) < cfg["min_sell_price"]:
                continue
            if not self._add_sell(action, item, qty, cfg["max_orders"], merge=False):
                break
            projected[item] -= qty
            next_sup["suppress"][item] = next_sup["suppress"].get(item, 0) + qty
        if next_sup["suppress"]:
            next_sup["due_step"] = nxt

    def _front_run(self, action, view, projected, route, step, next_sup):
        """Hook: if ``opponent_plan`` (their expected tape) schedules a SELL of
        MILK/WOOL/STRAWBERRY/MELON next step, sell what we hold of it now (before
        their supply depresses the price) and suppress our own SELL of that quantity
        next step. Bounded by our own remaining planned sales so it never dumps."""
        cfg = self.cfg
        nxt = step + 1
        plan = self.opponent_plan
        if nxt > LAST_ACT_STEP or nxt >= len(plan) or not isinstance(plan[nxt], dict):
            return
        already = {o[1] for o in action.get("market") or [] if o and o[0] == "SELL" and len(o) > 1}
        for o in plan[nxt].get("market") or []:
            if not (o and o[0] == "SELL" and len(o) >= 3 and o[1] in FRONT_RUN_ITEMS):
                continue
            item = o[1]
            if item in already or view.prices.get(item, 0) < cfg["min_sell_price"]:
                continue
            own_next = sum(max(0, _int(x[2])) for x in self.routes[route][nxt].get("market", [])
                           if len(x) >= 3 and x[0] == "SELL" and x[1] == item)
            qty = min(projected.get(item, 0), max(0, _int(o[2])), own_next)
            if qty <= 0:
                continue
            if not self._add_sell(action, item, qty, cfg["max_orders"], merge=False):
                break
            projected[item] -= qty
            already.add(item)
            next_sup["suppress"][item] = next_sup["suppress"].get(item, 0) + qty
        if next_sup["suppress"]:
            next_sup["due_step"] = nxt

    # ---- layer: budget_guard --------------------------------------------------
    def _block_requirements(self, view, route, start, end):
        """Planned purchase cost and item reserves for tape steps [start, end)
        (six_day_budget_guard.hpp calculate_six_day_requirements)."""
        tape = self.routes[route]
        budget = 0.0
        seed_bal, item_bal = {}, {}
        seed_need, item_need = {}, {}
        hires_by_day = {}
        quadrants = view.quadrants
        for t in range(start, min(end, len(tape))):
            a = tape[t] if isinstance(tape[t], dict) else {}
            for u in [a.get("farmer") or ["PASS"]] + list(a.get("hands") or []):
                if not u:
                    continue
                op = u[0]
                arg = u[1] if len(u) > 1 else None
                qty = max(1, _int(u[2]) if len(u) > 2 else 1)
                if op == "PLANT" and arg in SEED_PRICE:
                    seed_bal[arg] = seed_bal.get(arg, 0) - 1
                    seed_need[arg] = max(seed_need.get(arg, 0), -seed_bal[arg])
                elif op == "FEED":
                    item_bal["WHEAT"] = item_bal.get("WHEAT", 0) - 1
                    item_need["WHEAT"] = max(item_need.get("WHEAT", 0), -item_bal["WHEAT"])
                elif op == "FERTILIZE":
                    item_bal["FERTILIZER"] = item_bal.get("FERTILIZER", 0) - 1
                    item_need["FERTILIZER"] = max(item_need.get("FERTILIZER", 0), -item_bal["FERTILIZER"])
                elif op == "PLACE" and arg is not None:
                    item_bal[arg] = item_bal.get(arg, 0) - qty
                    item_need[arg] = max(item_need.get(arg, 0), -item_bal[arg])
            for o in a.get("market") or []:
                if not o:
                    continue
                op = o[0]
                item = o[1] if len(o) > 1 else None
                qty = max(1, _int(o[2]) if len(o) > 2 else 1)
                if op == "HIRE":
                    day = (t - start) // self.cfg["turns_per_day"]
                    hires_by_day[day] = hires_by_day.get(day, 0) + 1
                elif op == "BUY_LAND":
                    extra = quadrants - 1
                    if 0 <= extra < len(LAND_PRICES):
                        budget += LAND_PRICES[extra]
                        quadrants += 1
                elif op == "BUY_SEED" and item in SEED_PRICE:
                    budget += SEED_PRICE[item] * qty
                    seed_bal[item] = seed_bal.get(item, 0) + qty
                elif op == "BUY_PRODUCT" and item in ("WHEAT", "FERTILIZER"):
                    budget += view.prices.get(item, 0) * qty
                    item_bal[item] = item_bal.get(item, 0) + qty
                elif op == "BUY_ANIMAL" and item in ANIMAL_COST:
                    budget += ANIMAL_COST[item] * qty
                    item_bal[item] = item_bal.get(item, 0) + qty
        for day, n in hires_by_day.items():
            first = view.hires_today if day == 0 else 0
            for k in range(n):
                budget += _fib(first + k)
        return budget, item_need

    def _budget_guard(self, action, view, route, step):
        """At every block boundary (step % 72 == 0) make sure cash + the value of
        stock the block already plans to sell covers the block's purchases (hires,
        land, seeds, animals, products). A shortfall is covered by extra SELLs of
        unprotected shed stock, highest price first; SELLs are moved in front of
        the buys so the money is there when they execute."""
        cfg = self.cfg
        block = cfg["block_turns"]
        if block <= 0 or step % block != 0:
            return
        budget, item_need = self._block_requirements(view, route, step, step + block)
        market = action.setdefault("market", [])
        existing = {}
        for o in market:
            if o and o[0] == "SELL" and len(o) >= 3:
                existing[o[1]] = existing.get(o[1], 0) + max(0, _int(o[2]))
        cash = view.money
        for item in PRODUCTS:
            planned = max(existing.get(item, 0), self.future_sells(route, item, step)
                          - self.future_sells(route, item, step + block))
            cash += min(view.shed.get(item, 0), planned) * view.prices.get(item, 0)
        shortfall = budget - cash
        if shortfall <= 0:
            return
        candidates = []
        for item in PRODUCTS:
            price = view.prices.get(item, 0)
            if price < cfg["min_sell_price"]:
                continue
            protected = max(0, item_need.get(item, 0) - view.in_hands(item))
            avail = view.shed.get(item, 0) - protected - existing.get(item, 0)
            if avail > 0:
                candidates.append((-price, item, avail, price))
        candidates.sort()
        added = False
        for _, item, avail, price in candidates:
            if shortfall <= 0:
                break
            qty = min(avail, -(-int(shortfall) // price))
            if self._add_sell(action, item, qty, cfg["max_orders"]):
                shortfall -= qty * price
                added = True
        if added:
            sells = [o for o in market if o and o[0] == "SELL"]
            others = [o for o in market if not (o and o[0] == "SELL")]
            action["market"] = sells + others

    # ---- layer: room_guard ----------------------------------------------------
    def _room_guard(self, action, view, route, step):
        """tetsutani room_guard: at hour 23 the end-of-day drop pushes every unit's
        inventory into the shed and overflow is destroyed. Estimate the shed after
        this step (stock + carried + harvest/collect - feed/fertilize/place + buys -
        sells) and, if it exceeds capacity-1, add SELLs preferring products with no
        future planned sale, then highest price."""
        cfg = self.cfg
        if step % cfg["turns_per_day"] != cfg["turns_per_day"] - 1:
            return
        cap = cfg["shed_capacity"]
        units = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])
        carried = sum(max(0, _int(n)) for inv in view.invs for n in inv.values())
        produced = consumed = 0
        for i in range(min(len(units), len(view.positions))):
            tile = _tile_at(view.tiles, view.positions[i])
            a = units[i]
            if not a:
                continue
            op = a[0]
            if op == "HARVEST" and isinstance(tile, dict):
                produced += max(0, _int(_get(tile, "yield_units", 0)))
            elif op == "COLLECT_FERTILIZER" and isinstance(tile, dict) and _get(tile, "fertilizer_available"):
                produced += 1
            elif op in ("FEED", "FERTILIZE"):
                consumed += 1
            elif op == "PLACE" and len(a) > 1 and a[1] in ANIMAL_STRUCTURE:
                consumed += 1
        market = action.setdefault("market", [])
        planned_sells, planned_buys = {}, 0
        for o in market:
            if not o:
                continue
            if o[0] == "SELL" and len(o) >= 3:
                planned_sells[o[1]] = planned_sells.get(o[1], 0) + max(0, _int(o[2]))
            elif o[0] in ("BUY_PRODUCT", "BUY_ANIMAL") and len(o) >= 3:
                planned_buys += max(0, _int(o[2]))
        shed_total = sum(view.shed.values())
        fillable = sum(min(view.shed.get(it, 0), n) for it, n in planned_sells.items())
        needed = shed_total + carried + produced - consumed + planned_buys - fillable - (cap - 1)
        if needed <= 0:
            return
        priority = sorted(PRODUCTS, key=lambda it: (self.future_sells(route, it, step + 1) > 0,
                                                   -view.prices.get(it, 0), it))
        for item in priority:
            avail = max(0, view.shed.get(item, 0) - planned_sells.get(item, 0))
            qty = min(needed, avail)
            if qty <= 0 or view.prices.get(item, 0) < 1:
                continue
            if not self._add_sell(action, item, qty, cfg["max_orders"]):
                continue
            planned_sells[item] = planned_sells.get(item, 0) + qty
            needed -= qty
            if needed <= 0:
                break

    # ---- layer: clamp_sells ---------------------------------------------------
    @staticmethod
    def _clamp_sells(action, projected):
        """Clamp against a sequential stock upper bound, retaining market slots.

        Earlier BUY_PRODUCT orders can fund a wheat wash's sell leg. Their full
        quantity is an upper bound; the engine enforces actual cash/capacity.
        Removing empty orders would change the later lockstep market races.
        """
        avail = dict(projected)
        kept = []
        for o in action.get("market") or []:
            if o and o[0] == "SELL" and len(o) >= 3:
                have = avail.get(o[1], 0)
                n = min(_int(o[2]), have)
                n = max(0, n)
                avail[o[1]] = have - n
                kept.append(["SELL", o[1], n])
            else:
                kept.append(o)
                if o and o[0] in ("BUY_PRODUCT", "BUY_ANIMAL") and len(o) >= 3:
                    avail[o[1]] = avail.get(o[1], 0) + max(0, _int(o[2]))
        action["market"] = kept

    # ---- layer: dead_stock ----------------------------------------------------
    def _dead_stock(self, action, view, projected, route, step):
        """tetsutani dead_stock: stock beyond everything the rest of the route still
        plans to SELL is dead; sell it now when price > 1 (on day 29 everything not
        already in this step's orders is dead). Highest value lots first."""
        planned = {}
        for o in action.get("market") or []:
            if o and o[0] == "SELL" and len(o) >= 3:
                planned[o[1]] = planned.get(o[1], 0) + _int(o[2])
        day = step // self.cfg["turns_per_day"]
        extra = []
        for item in PRODUCTS:
            have = projected.get(item, 0) - planned.get(item, 0)
            if have <= 0:
                continue
            surplus = have if day >= 29 else have - self.future_sells(route, item, step + 1)
            if surplus > 0 and view.prices.get(item, 0) > 1:
                extra.append(["SELL", item, surplus])
        extra.sort(key=lambda o: -view.prices.get(o[1], 0) * o[2])
        action["market"] = (action.get("market") or []) + extra

    # ---- layer: terminal_liquidation -----------------------------------------
    def _terminal_liquidation(self, action, projected, step):
        """fieldbook _terminal_sale: on the final acting step (>= 718) replace the
        market orders with a SELL of the whole projected shed."""
        if step < LAST_ACT_STEP:
            return
        action["market"] = [["SELL", item, qty] for item, qty in projected.items()
                            if qty > 0 and item in PRODUCTS][: self.cfg["max_orders"]]


# --------------------------------------------------------------------------- factory
def make_agent(routes, router=None, opponent_plan=None, **settings):
    """Build a Kaggle ``agent(observation, configuration)`` closure that never raises:
    a failure inside a layer falls back to the raw tape action, and a failure even
    before that falls back to PASS (with hands padded when possible)."""
    chassis = Chassis(routes, router, settings, opponent_plan)

    def agent(observation, configuration=None):
        try:
            return chassis.act(observation, configuration)
        except Exception:
            chassis.diagnostics["entry_fallbacks"] += 1
            try:
                step = _step_of(observation)
                player = _int(_get(observation, "player", 0))
                tape = chassis.routes.get(chassis.players.get(player, {}).get("route"),
                                          next(iter(chassis.routes.values())))
                if 0 <= step < len(tape):
                    return copy.deepcopy(tape[step])
            except Exception:
                pass
            try:
                farms = _get(observation, "farms", []) or []
                hands = _get(farms[_int(_get(observation, "player", 0))], "hands", []) or []
                return {"farmer": ["PASS"], "hands": [["PASS"] for _ in hands], "market": []}
            except Exception:
                return copy.deepcopy(PASS_ACTION)

    agent.chassis = chassis
    return agent


import base64
import json
import zlib
_PAYLOAD=json.loads(zlib.decompress(base64.b85decode('c-rlKU9Tfoa^!#M=lehwSw(W+dZg<a!7cTm)iU8Vgu(3YEDS8XSlE6u=D%<Ez5J?^kr9!n>PiD!zyRW~d8^L%$&8GQ{Ev74<In&0mv{f|pZ??B|Nis8{>xwf^~=Y1e|-PrkMDkddiT$N{=fh0-@bhD<>SBn`CtFfU;h2e=YM+lyPy8?Pk;RM@!Oxi`TXwb-T4pC@4x(ayZia+pWglW{NY3UUH0>z|N8X!&L2L0`uvCdrS_A@umAFg?>;@hb3c52%=;g{`~Lmc7ku;a{O)N8U;p@r=jT6u{fDO2KYsja-s&%d_uV&t|I;5YkNWzc%V^%a`1oo5>&HGfZ@T^PwJEb#K0J=S>vunW_u<<=efi1fpT2h8dD-ir&R*6$MRxGR`){6&{)PE({uICG{QTqRmk-~(7hSsi(jB$igWGYBd*R3D=WoCK`}fZuKK=lfNIPD<1z-C7!{^)6l5d*XQBB&mO}Y*&SW4h+*QTA{e}4YdetH=uGVFi%wI#DR-2eFg<m+HtYt`!F>w1{I_co*XSl@p$AEEN3$JJ&HZ2Zyd`1Q>2>_O)E-~TP{r`db2=Wyuv<pB@xyB?8X;IF64V++E9^4=T9J5F~T=Jx$+n0JL?X7e3~yZQAR;|}JTjr;4b%P($Cg|`{_LBE!ax87*%DvTWj0}C#YHylthp_c}z*A@o!{%mJJ!IiXVK-*W99!wtBeCV=;=7XGBnA*+(*B_WheZ#a3g<o~t*!N~#{>^wwynk<hd-*5lAs;_{c>d<|pZ@Oo)93F#eD~kmBkuMj_`{093+(vhnODo8`5NxKi_`M((=Yoyb6mpZ^8CJ?;rmtE8mf+mmRW(>=<=Q>uLfOq&YJ6PjLt=Rjv5`z-xEf%?_9Y*=53k@_V1$h-cFj=wY73<@2A6(vpU&#3)VS#*ZX{xb=)qC=4R!aF3#|8O{w4ZPkV((g~MG!PV&B`c#2+Q@#<xcHoLeoV^Q79?0xdmM?esJo?I7XfE%}S@$e#qM7GZ!&5J=$8ckp(E^Do|T!}lsa5R^ijI^<`^9b$*&AmSG5AQ$yLm!?&M)|f1dYmt7xNpr~;KthX@Bj}v&aAYN7cOsL^v>>p+?dMe8{8Co9C)~S2MQ#Z#~Aq0*0D_J84U14Ijrn~00qA+3S*wg0K|5#Vu-vgB2h3+c4a6++9E7SVtuMb0#a*4+{hsuSU~@)?<$^aYrO}-MDc>jKJ-lkI5H|XZyfX3;QLt2E}6bdQq4v1&J`Ao?z;I(+x=Xbr`Sh)&z8~Jc<nt#u5h>7sH}MvumIlu6B+70Ya|%>&2WMnVqLog_03u`1(T64P<;t{&xWXBmm#$CU&Rg%Xc6r)i)guFiTtLX{tJ6NwLA>mz~310b-o3`tg!oYQ~rv4zCAbB!3BXx^0tm^A6RY1i+aqsjH^;M>&AYwt`Nyt3tFP~jvo|R)5_{Hu5L)M1!72qEDn7hyXiBonkd@3C`oRz_F+!oc`b}lGC=?a(qUGWm)&j5fhEF?s+H~8XozG{lOFp7-QqN0=0;+O?{S+^8dB>$CIA*dRtf0H)4J<tHj^mGN|%yzg2p|VtPnuYS`f=5xFvXz0rNPqSN1%&;dKzc&<lc6rq*LmC&+ug{q*sVRl@{$0AoLz#c|MOVe;PB%Wcy|@d4A&*m#2q8{c2?voIG2OtOh)Zcm^OoFdmF40b6wV^#tO7X|5Ve;kKsIqH{jo2*-m3^GDN78`fBBPUxdHOu+-xRy-eJP+_z?|tU0DcR)@*YuB{KfOQy?)lTFe*{Y#`2gyD%N={rUw}@C09=757U9l^LGg_F*y7t=k^{zzW2kovpPTZJb6l}Tyr%oFvI47MhF3vTaystgZ1))`48!i6<!mYXN8DuvE(~Ksd!&O*m-J^DQ?Dy}Et2R;X>rv^Av6fsEuNB-i*(wyY4>@D@e#?AZd!i3B@KRcX^Y(+oB~2m&=f%DLTx8L?u^tNdg>&m$GsUX1km;PZ^eFsGwg~ofBg7m-Au{*9&(=W{oi`+*n)m}$9D6O1?$niZ@H#l4v}wFKI;9isg)6j;ywjQAC>8Idq5H==pnur`w#*XAi_iZYMb)xn2M37&Kh$wm_Y6<aI%NaJLB%-C!kW!Tr$&!wpQBEED)0O>>j_0149!)9aB3t4V*d-OM@jbr@0(Yz9$kJ3J(32TW{DOD0Q?iJp;h)H2fg5NOkfH>=WI9>WD*A-JEKs>HlPLG?=T$Ek>KL;^#NT(=^SrNo{D<x(-Gi@OSAMA;O8tEbUw-rC4XlOfi-X*`W`XW6eQhNt4vS^tm}L7<ek23*dl2q>!HNHn<a+Ca}t>qtV3r)On{Y&Dd63F?SO??nweDFLoabY5Gkw4v9|6quDHuh8u+LF^en=IvAnRe_}?J4(t1u7m6m5Q7R$O5hH#$MV|RZ+v~sqv<};`_ZS3%V(4G!SY4bF2}t+)RMEc7Mv8k*Pv(hZk-L)-j808!q^<&n1A{kSGWpfA8q1K4)qVLjuFl>6N-1<^GcA1&u??ez?qp^*fW|-^nXTa`3l{3NW%<ZS1C!pP=%rNlN#E@?ufQ22y7yjZ0Ofq8so`GGkqsNS%-gfJ@$`#H4a~bhEou(`RVRm8*}!QOgVVc^d_fo;-2=@p1aYt?o9nsewCi;NCl+Tz-&uq`4Wgr1B^z$0j?jx}F-Q%3#josww+v(fiCaau*SW}wIWc873Qz${=Vli{=;@7e@;qtD8R>||y3AaSGJ9_3yRwT5{Nruk<nV1LIfBE$k8K8kY#hi^D{dAKdi6czc)4P~|L()zUm{qVq>|6!>tnV<8jPBdw|vt+tp#vm959CX0G=FlPE(iU{>LWLy{mEQlwGX==NKuwcd4~!dLVLiJIC$AiQx>Mld;j7<-lTzU^g@bph^I5r;(lLU4rX8B=_9sk0IboNJAhh4TdE8;M@@|$EfPecAq*IdjXCEA$v;OO3*BPx_d&v{TtOxv5+=AB?m1A+wM&Q#3?mOEjrjUOmR0n2DEeXqG{BvoEHVAfMtRPAY`qz9T^XJ`Y@sI_Rp_Fy@u{=u2R~}9uWr4=2_c&ly}JXkMWO-WsE?|7}0jp39!S@ojgt)=SuH5)+WCNQuP@cSJrq?E6*Z}qECo(QDuPdf6;JTkJy}MVLqt@7BpEwtj6PVCeBBdJr?()(Z+$%Q`?B6C(XH31J+JF0ajvkoa2N=JF6&z08?U>=cIId*YJ*r6&PI$<-tV|Pnlc2S(=1rc|V!4@p`kCnnQFQk15eZp|L`rRVvpZ+0ejwoHKv-TspF^dI%Laaxtc&(M|)VfUvG;pq9*26A?U;!z(?!dqy+HFAD+n-ZBzwfHgM3gI66sc_E7E*>sgYEMAD}m@ms8EkMSoeTT#cqOC5^C@?Dcq-K|Og+^OnB_3~4v1q{Hblr#Ua;HHa$|sfRXMxHJ^NQTCHIeMo9NIkUW=0$tnnSZ%d~UPyFK>(Ic3m?ortamz^q(vu$=9t_Va0a=$Hmmo3WD+LWCt!`ev%Jt|M1IRl==Jp$B#cg%b3=#(2Jr_Ds@E14y|^0u7c`_zf6z7(c@!yjO^w%fIv$rhw63Q7w8+|%u|*az^@OPnn)5k`!hln{Ga_6pXWdxLbZ>iFD5~XXTl6^4LHL=?a^GI>{K%?f}x*|VnDG~I7uWT5TX*&rAFI?=;D}$@bCwCW9|95JZ7bg?oUp#u0n(90HHsi4*?S0X8!QC(B0lYx=F><$HsxJ$b0CW=iH}becTQeCIj+aQL8Dxn5CJw<I}ffN9!)|xKFRfiR_e30FQK{Yj0#<XT%{Vu&rYU$@%u0kNC~UR$4}y>XA8g89(C&A5908OG-3N=j$}53IocUbYyuc;AHWN#8oz{f=Q7l%OSuaE>NA1-nsEey6{;i)j2bf9xQeLaL)SM4)BP-pNqGiH0zL(LjiuGI>!3*+SF4w&pxhzYd`OYAqynKEJ={NMhiv0j@$~Abl_^l7+jLs8I@fF>XwZWFI@1nTKU<Se`igNN*8QCtfNICOW0Bst?+LkgsDPDo%L>=54FTYi4TQ?Us!R8)Xi=YP{eYCkO9laq95PlqC6M$NRkpNA|RBC3}Q6l8>V2smeke?Owt%xemwrbt*rIBb~!^uW-H}v4|31#3Um`n<c*Hw{koF}XOmX5wk(Y;BTB@VWW@I2W)7E1Mef50G2c}y<MA8yTOWNUzJx4+-G>B@<y*+q-s4%ViMAZ$pBHq7+(pK!QTZ?^W#z~~bAe;@w1nFaygKCHgS>6~6w{aBYJ`7)rYrtBt?Vr1SEOuh*PMAav0ogXC!R)N03khKI3)9cx$L}1S3LP7Tf058`*|%xRgDQtv+>Qrvd+{gFZWZw7MRIFmV1VTxB62t7>gc94J@XEc>t`N(=V135JB%f$5?5>+`l(-v7I%A)_~3ee=<|1YZCT4V3`+-dr}%6I^C#0k^L+S`{3g`Ew0#8b#f^w%A(e(g(A3kLN3nCdW&uZ)kV(xV4Ta#^bFW~phXZWr-2rzTA+AfY|5o4!QU3g2ogLP>qFCW&<{m&9|BYMwGb0i>87%_m2z5O$42uIvpm3^Vh)d_Fenx=2ub<BWEV*imAT~<RNMhCA%Q9jF~!acGry9A4>I7xVvGOelUCt+2HpnE!HR^K^E$OC>_sJQS5lN)uXZ>{axt;P0GI1<4Q!&uJl7$iXA}*884=>1ThThZjQfw9(3U|I)Y}|m?z#Kevui=}b7nm&R5cj*;mmeMsWtTqq<{q&4;Rp~uG-OHQ0&E+@g)oST5q!60AAsVcPSO|HQH{Mwp9Kn&5v0vrwpX(8A<X~RiaDG4_wFb9kGphW%b&gu$LT*5}9sFhq}2;4e@tG9>VVs&<CL>S>2qqdo}Jt53ix$l|)|E;jLUtz%7Y!>3m;c#0bW*OT|DLS%M|eplCzZs(o4QP9@4wg#|jUhWA^u5h+(#WF2Hd<4!A;<tZSxo;W-uFNx^$0g5IV3tRuLnB@+(g+7&)pGGfFZ#Au;?;IJK(U-gN6yo|?u~8NYf&~v~D4ud4N`xT5yt5->b#W1`>Qv`9rvA$^U6CsjX`RaSd{@xeKvw&>yh^<wHkDnrmuc@#@^qlhk#k7|oicu$#!iza{aiv&Xq$v~>_A0x0xgj2Q^Ac|UL!qxN`M55FCn@AF5HYsdUX2QVhW#&Gp&(5MiFCqpF9%5M@IOHY|+<00n~|R(Iphr%06&1?rCtx7ijORaa?4>6atjYy8H!Wm2p*rV>rz#9Yjn7{vKu!LqGaNs@QVQ`(TAN;aX(<rvfHKF(ffL=i!W<yg!vZ{F(8gVib89v;59uMi&$-a33)kO<CSmicsWcEm)$#?a&%<%7|qwk*Gfd{4$c&(*#?m&f-URkv=P-^AakKQ&>p-A-&jjD_F3M5&&sH0A3?Du1a-`;Gp7onfG0`irD?{aDxgo^`=AX?heJ9_cm0CblziVvXB@TEuk)XEmndiN17uf><vIoc>sE<oPfLF1+_hyxk2v+euv&h%i9ZP;mIXC!!CDhx;IMVy(BZhvxBKv2ZQ<LVk@c#rt{0z9{Opqew;~awRt{%MTNyxz&UcnmYPR6%-mOY)+)H<wzjlllyRKhXFscln4X%r9;c6pXvyEzdA9XGD)_N`6)1iF{qHa4<S)d%4cVX|I8F6%Sbi(;P7gGb8~dyKE<L<lL2jOk95ng7AfqMfQZe9-9O^+ek4LSIXn4u4Arw@yu4trNwt`e+ihgWVt)LKEIJXv<D|Wr#C|05BA5qO}Uu;gEe>LF>7<7joC5O1KW|vFKWEfpR_>iR{W(p<in2v~Zk&8{ujj>8Shr*rD%Bv$AEWX00W<R@EeO$`(u>fTlmn9qc-m2z2Z7b-eS+q7^3c#fhkd9|2leJ4S#8B<6I9CjudFUw>w(52%fm=qCWq1&|hviAYF?mqi4l&bCe3da9i&&#yB<7sMfIaCV@!0qEOuqf@_h18A8a*BM3Ej=iS(k!neaU~eWXybc+;ZLG>=BhXR}XQ|I|1l>Y%Y2UD4Z=-h7Vv`pCD3q0p#@LRWW)QG$F7UHwhmV1&*t8smhhexxT+0a|b7D4hamaK>{gxnrfRZU;+CT&XG&bkcdqGYi4<EIsk_nVU*o*%>9L9aCbT!IG}VvT3Npygd$S3ItXc8{**`g?buzRfd!%tK=Q#oQz(2xwDzim{x~!!Atrddpw+=)sp{T~f(Cx-u3flLF+%LRiYTKNSMq%(xd0_Nh2?;ExXM(y>TF^pFa=l5;R?KH9_V$(UeFK(G@<E_poJ<%Bhppk#%1fwsBqhhrk!P?I<{4<0J0-l&}qx^NF!d;_7Mb@pY{yhDzV}~p%8XVP`OEwKN!(_0S3H=Eu#Dh(f!alxmjYu<1%4?Ggs%BiuxoK48<j`IBZ?YPV;o>VsDCaR!kF#KB3g7*Va#^>__IgqlG(ZHw|TNkwY?U3+s@CA@RJyqyVX?)sWZ06~xe)3g7VH@hnJ$=lMqztM<*fUd#oB2hf4)(j~4SAv=K`12U5?BPS;)?o?q<q`2Ml0#?OpY>CxU3M!F+vjem|l*jm6P(WkQLCiS_#-7eMNI32<c+-?57aSkHe~`(FJ$Fi|u&cq5dK;t_L?bqaieIjcWr03MI{lcq+dYkAfeU!XvEQY2gjZ6^V)>ws6nLNC)|xAa64D2*xSqhWA1mtveWDZTWKCv~5?wEJjZ}B&=CWK#>q5_Yt!sD>G!-Bo3lKgrAV5Dj@(OkvQgK|z5>;S+FhFKr*b|^(RF%%8QPU}ygeegPRd~m%u$B})7};@0ac-uQ#wsrNE_5Zxm2pzcJQR`X>x?C>*y_d>O3J&*uI!U1U`605{6v=tIZD@PbZQ~sn=*4tV$i{o-|jB|z`Svq!Wsh%6(*(#e{k_^n)qFcD^$ej2=J6*9aJwqTF9#6v48hXCcr^(f=F6Uab7OTn*we|tf{@vA!JZZ&L`|LOJkuyle+g_9N}1(iWE(gDyvGZj+OY}w2t_XlJ9c;lRFd-Si|$Yj{+y-SA{U*CIVi1kl2OQyUj!eznmP2Ys@$v!y3d8tC%~BOlA=<S+KlB24V9>?F+)u#6Uv$)Y~ei8beigNd~oUl3?gVnxiij1;$mRD7;n7nff!PCYu9*b5aTq45mO-QTM5XE<v@he6NOuGr0j*EF;FL9EF*s1uM!$v)vpXnNrOPR|%gE@>Ga&StlKt#=jXvRWRw#>kFK5z=2hxk|!8vT6MBf=*E--m?XV3yq;;yBoHzc9(&fld7n^5s(ZVx3<GW%Tm3AKbu*3AyS=NBpRqfQJQ;!Oy6G96`;pxpqP!1~WRFlo(G{H%xXp!4by_JvL!TE4BT^{}o!WqzKH#)KI!g>1>OA~JML0;I$?`}eu{Q=1P!f$nqA47QHC@k~0t0i73xLLxs?wy6#YfAYM?=_A<fpL3z0T--$vJ83h@npgC2o--B&y2Y5D$bI|EnS-vKE2CUVwf-$Nn|M^a^o2!W|i17<|7gW(@_n&rsd4gpr^EpYN6wf`~bs#nknXvK^XDM@6)h<P(irI~AYS30@*Lt?X7aP{;g|#R)wuJy~2=&=V>s?=K5lu(V|oQ4?w-p2}Q_*de_$aJ$uFvz^LwU=!(=6)al<@q(l(CzXn&&t;ZKmr+Hs&pg#ZvFkG8fK``Q?m;)dIAh6BE*N)m7`X*EktpmgA*IWdcy``ZN-_43iW!^B(G%^-t*gi=Ghmot1qIr328GXI52teBZ2LS52?wEzoz6>*EbzFAgNr%TG)!0pCd%`CO=T)7F|qJ8V}>9Y%k)AYozR(gntVV0T$lh%2}0sraurM1tDhLfP6(z?Xe2HnU4kV_4LOhmq^!9k9ZN~MOXm2qpKc-vS&@H{`<GzJlg;PItx~3iIgm<ry<B#}!xFY{jwR5Nc+fwccy0)*X~^|9+=752vt+3NC{_wFSr`ZdO_WD~XPP?d^%9KdXM;Rq<xx{%9vai_h#4w2Fbug$_9q~>JxfJWX!0fkRw8=B8K2CX=LPiw&?WL^!Qq%GH|OeyyVS_*@xnX709N0dkGrVpnS{Uke1ICOM#_m%BKmn5U&;!HSG|7dMYAx!T;-FF014(Zarw?%fW#&ybt4lti0Y?2OE<Q!<O7X<mg1mfeZa`*I>km<COYHk(>saC^Y^mu5^p=MRH~O&Nd#NPS@1IbGs23LW-xO2eDt2@t1<Q<c5)d>(vvSNSxjg}%9(!ziFH|;1l#E-9uG=yc@!BzPoV0<SwhDGl@7%9M=Eyptk+ahhX}XzCNbsdhU$DHJJ6dILllAsk&Jj4L!Wc8R^pA1=IZcsR_cTfd?&`S@<?yJhq)qyQ+%ZWVf$zcl7Hj08=#iJA(>gS#v_a2^_9afmk`5icW^xjgC)_g>d{^MG@fYm9-?Pyp92&f36bla3u_2=bE#FT6bQ+a6(^`EmQ+J@Dk>eFDfFJlOQXwmosQltbvWRxdch)fdiQPAi$SJYf82#^D-_)f45O|(>~5v%b|ttMyuB+c`z`a8J$`xkPE%7IPC^e7!PyH;j-aO&yfzO3ewG|P<S;$2wfsFE2hAVpA}+01yuqg>U>rGU?xXAcrje%(Az#S9N>avHFhBl*eL!N!&e=2%<QLSK8Pk$4E8_6=+)t9iort^_kNk7729dE@<C5>w^kpy$QF<kT@AYd{_*rAT$Ve)3xvHv-$T698+1{L#%X|OQFMYeZnjvmI7gE|t?5YPk$Yn~F3ieb>ERd%7iK&)y&nkNYVk|0<!i12|%yp;;{ea95S_M%vv6B6t371raQOgP=jdl9g)R$`E6^oRylX(SZp_!PP1Smm8Y719PuuIvx$t5C>r2s3?3xFD5gj7qOVAt)~721@_BAIC_W<4VG54B6DkFCuBhf<0#F~a~EwjA!K<GSFckk(+2ZYv%_#EUGlkm0b+1vvNltwE@>J(=lQ)&dN9GnI}!tjJ04`JqTNwbV+_0XMmqPd^vjs<_CRb|+KHTMgzY?F?V*hDW<igRq=R#2tb@SW<xtPOcWycpxEuL4kuc+On{80bLy`Bt9mSkv0Yev@=qHv_QuawcHc7u~gK1tRaX`$lFC$snR;?au|30lnO!rcs6S4_JxuSm1}B|QlhjR>#J*=oiT=dKO4(v8iPnCIX#$gdZ$A!HM&sJZeCc>tHVOPuH#(2Akl;%88Nm{xkt}@G6TKrFxLH{t40%iYg)lQVjd+MPJ2t^FJ5~^35&XVocqMJ=4<<frg7vwD*_5<L+GDfb)3=Rpy9N6jPu47mdM>Q!^y~>_7oJZD5LXA9O}k5R1a68d64K1FpL00l=5XdB`(hHIN7>)vdpoW$Fp4$yJUhJc)SI->0A@-aOAsfR=+XgqQ8ETuTtq7WT3@We*IN}+T_xgZJP4)j~$ia)Li)9u(kN9z}vZA@`=ulSplIpD~+T*Ft&#pda6v?y~x4dybnsGUl1f+2&0IVO$#;;mOw?iGI^MIpZJQ5)oz9%Tltp-V#OqS8eKTEQl=e?9|~SiJ%SryO1I*9A!Q~mn4JAzuLwa?emtg9p8^NJeJaE~szXk+AFm<G!YOEYgM|d${9J5ZU-NLG`sXyx$in|lsX1BVTV!EePlW4#@rE+zoStYm`cXg>47=MI92-0#S(Y?gnw`{N)jLbTh@w)|#An4q?!i2+bMq^xl3ou;p(wITaIL5nl9&1Oob8yYQUp8zXM`u?0FBPjbze+n0Hj*{6T2!2OFGV!Le6>2UntS0`O5WxC~G*L6RD~MEG+|5=}K>8n(tXdeN~NGwHI!6KH%$k4G6xpF439m3TH@lPl^%G&Sz>oD^#1HHc=$C%!j)De{H-Ye(3aHIqYF^{CQDj@}cItGB1QBGM(LQoc_oLb9FBRf9&^lYi-HoBY+a7UpHD3d?>LpFPmIDHR1u<bq+I*nt-ag^V;uWyZVt@aK)~2NL}5u17Du{=_5KvuDkQ{0~yrkrO(tFg^sYL%G=hEN;@Q+uQ_k-=*m<GR;qRhok~%(z{`BOB<TTo%~|{;lQFXlc%yX&jm5hFnnOXV#Ia-9kzhUO<D=REa%EzU)qD&0Na;6Qcnzybp!!e*6$AwEv7e{*ga{o<S=bmH3uDWwyBvWXCC~}XZTpo3vzwfR<y4XtKIx>AEl#Hp-%LL*({i0Hp6xG1HW_l4OcZ~Ca0{>dl2&LmI>;fIp+MFa{y({X^HM594-;k3>}|*}V3a(Lc4l4gXe%hPLbA~dI!#%@-zEAd_^fQoV%NlN0UjhnnT`=zs6+~oQ;jY!c2oV0h6!ocMwb__X|>O#>kC}33?O+j@+Scm#;yFoXg$|eGSouAk|vt$hl>eR06qnTl&<m^^XY7J2pd0`D?Y#b?RG99s3|fsB}obnb=pW8yCge+6-se!`Dj3`Fsu<IjOrlZ0u=Yq$^%=(<AOK)xTv|19x&PoaDEePt7HHR>{}Hp11JG$B0bHWj6CMg;G!h)cX1u*L4Eb}I#-SfgtAQux={$G99b=xPAsfY*oDgD(tp$wb6Ad~%8WRcb83<tJK<7PyRv$mL{?w#HD8BjpuR*xtB<`=hHgQ&_1dSM56(O^#8p%d=d=gBh@ZO*;X?p1PJhD~O@bn|j1r?dz{-@C6z5SgV3t<Mo^e}LwcTkMsA395Fp|aDK`qYDPDOJ#GB(z-LSJGv;TYx<7*QIU{AGIJ$IAfUfR^L3<wp$30!DTzNd};bQq85Z2J*#yqC8wo{H^7pzbH*NU@r-6$75bBahA=}AU*02J<;oeOYQJ45RhdoT%2Y_7)IS<%c#a30Z@RUM#?`MsSaPfqlIh$rWM;odOv%Pe{K3%HCYx#wF{OTV|}uHHX;-P#h}H_VM_c`D%6WNdgv}TfV9@_>?!7|gsk~aPv3(rnH;ikiApt6OQ065SC=VA<^Ct@M}1dDt3o;LDxs0tP@W2#A4@JNMAK0%Xopc;#lv%Di2TXkE6Tj#o)uTDe`ZSfPD%x<bbPi>AtL{q;2G$cbO)2?h4~J>`gd9;=XLT;9ui8SAV;N>JjMPVoM1h<R;z*_%al%B_Q|^0K7K71;NL_4Qo;Zpevk@{GKH!a2qp8o_kqWCn6GdRs@y8Dnw~u=lfkR_K$c2BA;z!7xwX4iE@8vyvAhf#lax)eik|wFShTChik{wM@+nJ!t>7Nb5TV8qv5NRj;`Ihpp}3IxMBt+rYirO{Dzh+B#X31QrBa>97J8s=vr^3!q2t#_`nvRPB>|*5T4I$mBD1K&?UXHKq*_51?n~)`ThQZ=fL~=P0Hug_VGjZlQ9RMHL{+TB%%t#FBH5KP1>*)Q?N2KSC2%yt#tUj6vO1PmP^U*|lxTRcE%acSd-Qecw5`BXph<%DPkpk<oQ+e6fp8r?QoE>+{%IzoQ9rVfq8^n6D9?^p5dt}J4!UiIs3l?sh&|?=)iX{v9(0!+Obm*efqlIb8NW$ZGNBoDuD)xIXa!bO<@@;PeL<BihF++yOqd^s8f27_*lgke&A+^zx?jKU&sz2pxU_6aJyyN2kF7sFRchM|n_r{04T)B#Fu_zPENn`=fehR<O{$rQ0VobF8i`gkDfSd*CNmCT9Kkwo0+%JsX*ZF@5_SA&b>v7~ls5~DnLs1|cRzji;oCob^YL;ac?FUqJF(EPatSY69KsbWAc4<8TN0EP3~aH?z*j8pngf8X6iOfhga<&XYW}UtdvKGI>77$BlMr#XF*5o>uBB3wB#{J+>>4V#*<}oT0-9$@>CA?$I|I6oT}uHOl+i=HzUfXIHY9<Xn`@`Tc9l%Ov*As|e=5#Ci-gL`xC=#>I9O#{X*@Ve>Ho4;wA9P7WqB8Lkd?!xrNazJS&5<L=U=@aj09m+D%%d3mr4>2xKhX)tvN;>r8z&u_a)~~z4^dgX5-4sVngp8RS#ctE!9PyqDXY}R<|GZ6yY~J)6hwF?HzK2!L3MZgs;14z^dbw-r|(KMD(w(WEd7sIeP0&lt#xM*V{hRKaC$|f0%T+ysZdAav$EJG}g5-QIdgtxcwMDYT@qc>ZPT$dr?!F-W=XJTUm|I1>?|BIJsG~JBpvOIzua@B$aCcQ{;xKDp|tO4Qy4IDP|0~M#DMFDyZS{Jfg#*{(`nVKYpvurerFX`3T^GluZ*T2sW?J#Lcjo6gsv-Bl4!MqKX{zaC2|qM@%iauj5^w{-}SU>6x{b(-W8WDb+~@=7eqYaWe}1X>(84WL{ZibCqNdyOd%{D*zYk;4Bv9@OpFZYu3D2bEG1V4EEH<&#3^;w!&>uf%0zWI^ProkAf0iBEWcQfX^Aj?gHpzv|?v$v6uInx*XSx{U=XoPYg`~gcNrx#r9c7ZZk-|Zr1(l&mX^k|M_G1dM?K~5Ai;K`1k`c_(d@I0yGIvqU6?E<16&xz&!jm#MIs=Qt-cxZyaAiSB?j}yi417%C$iR1^BNu<iMmS;0erzG6y*O@Ly%5=Rw5RKSrta>_P*NNO9!fW}r|sik%1Z=ubQ7^KX}pc`~|qDjwea^IyL_?*IMy-~Z=7|K&@+{^Q;D>qEEG@t4Zv`}+Pb?>o8Wy3Gu|Gu6_>c;EXUzWe_D%i8<H^Yb5H5xuXCJ8Wu-)_&4pHOY`x;Jq|<>q^@gB}ynC3%q-9a|KzIl2nN%@d;nvyz5KlQ#FICDm1K#`-SqeASzbR1xp^nz1$oO-=>o4mrjqO(~9bL+XSzBvY;<+yX%*hPVmOfR!}mIz}*fvaH{%IZ23-}TL-_|Q4GG`2VAZhv%D!C{-Yg@$o2u_XQmmrl=(Q%HEX3YUz&Yd5=3G7!&f%^!&_6J{cB5(o1Tg`V0-0K%B93NCfWLsX7tkZZO0(8xlP6{O;G@&IrNNXZ@Lk9|4w;YxjS{AN|txt0@kZc<i1Cp0tkGEoDjSV_)N;pdmQaD0svi6Yb@VAV7e!N;8djuM;j<{CCJPkgf~Rc<@{`ehA@ccytLrh!9oXYG`cg1e}yis<EejC7~hBbd-)`tHuexT)Ei@2k*uz4%1+yU7aA<1*!c9t12=r$Kz`=(;+>&;=*Xn(bk+j-WG~>;zFwMyR#)@xm&?n?N<9%$*{68BUEV%1gl$+*%ak6wS+Wny<sBj+=`rn%Vi*G_Gi?zlbX^<=nn}2K{2|lbv?A$+^}V}x;aIQ4{Wx$nTUfRf&f%@CC$|t6^(m}zL`<SD-w&J3f&yFS9ZMR)M`+k=*C@Fa&1AFqB_Ui?%+6smbQB>LY!BaHPGu^<us`X(?A%2wE$6eV1<1*cs=@p63a7bmerGBJe`)LZM(dpk6{H1fGcBf<mQDiZLlwN9Q`?xagX*Jr5KA=XdZwnPAe5%Z>QO8@!TD##aC#Wjw#BKm)NIIwbB&0Quj%s)+fChqWz!qo902KNe*Cai!I$W6ovle$iNA>_G;CEwhCWY>6|F?+(MMm}Sc~0&)2lXw%6%NRdqZ3*Bq;6dag(TkYu^srQ3WkpGkq10+i&Rp+R&*xOJzGOEZb{3@55GxOQ4rdO2SAf5-+R82~rQ+qsxuWs@p;*-=pQeG~gr}5aYT)$KrN;#K45#lwqSnt!_zATe9CoV=nF6_Jx9_H~r>_(J&0QJOCXV?#wEg&6k$!-1SC<|3Y{jM#4x6_N6s@b$GBUp*;mHyg?devpVa7GL1biG40Yq9r|F@pIrDoFHibFzB)>#dv1CThYA7JTZAmCDZJgbbw^i=HlEAaM%UFFtfCE<=AEPzn>eG1U@!SV^ZdRxY`5v$F*JXOE>7L1t(~m~DKRnB%F;rV|JW?F-B6GU<-5RNV2$l>w{_=kiqygvn}n*zlncLI?fl)|UI6vo;;qfp=iQCcTO#yKj}VY~OrSMW2XP|SRP0!ZX^ReiR(Qp)qzSOS4(0AxXsjliKfYq@7(v+9@!D2-h1^S1bojQ;-0rt7ZJB&vS@WEi3EkNJP^VJY8nwWcg-Gr7bl&flkigwfjO?5_l>7Z6Bh~!`tbA9&^wQ3;PpW`h)&(4r)f55EoeF85R2q`xa|sOtUhedzl?Ru@tTf9QI~WwRv@0)N+Ig(E)0CErqs1Z?h2YHoWM9`pxPWU42tehMQ8a;xz{HH}?_YZ{ZZMbR_Ebm-1MjAS2j~ybclS3SxUzKtSpSAAd^>K9%4ne>T;m4)1wkYg5@&RW5NWmwUL6M;97X^VpKM|r#7q@h8YiyM{vOx4Y?52fi#|wC5Z0}okL$w0z;5HXw+lScj4ssuSGnmg2Il9V0tO_{uM{qK%Kg0bY%B<RXUf=o?~__aC>>0s0YRR3aw<>utWgebB)*cOOr%0WYG8~8kQkQ<YqO}3>+Y^%EPg!tluDAU5W5OQQ7P?Abx>CT*bd@EP|WW*{UEO@Ut*X8WnYHP5V{jMDFQhw!VVT6udhbI$oXK|9oV{}`-A{*O^Yhk+<b_ee%jT^`h6dY38G2LLAW!I*D>GML;Ev>3`HAv<w!nk<47osgED|7KRpN;+752AMk0br)%voCzrA8Z(^SCf6~|FhA1Pi0tY2U-W)YFe_Q^S*0qCiZwx!p<0uA+QcTGn-j85np-~>g1;x$OijX=ZvZl4r;StiSgs73R4inT$^1ajw32H#mqK@`HJN#}Se2&OE}wvAkohwUOc*d;qE2S7_p^f1^xu=luH!Mrfob61M(r8hf&r2*KuwsGX5mD2y(LNhj(kFr<F1Tw|gWSqQw_e-lsUs7FNsR>S|3z_EEor?rC`WGx!ic7VE)<wdfTd&G){lT9EWO?MyiE1+WQ#u!yb{~C_z;I|4{1TRN4M(QV$2a!(siW>KE0>DMUoK&@xIG$qjjn>Y^4I~rXzw4iLLDODacfHt(`v`C>{Qz_wYhWM9@o)74zB%A83&r@VHZ>~61|eH?EK<ulFx>lOcqa!rGd4x?+{A|s3cohv^;X3XvJ_GJJvjCH@nf)o+pCBmqyBa7XWJr^0lguxIkRm+JjmO7S)DfTgJ~FPRWHOv}-b;02aZasu?VGSU2Tlx82p01v9EmiIG!BqArcyi^j(5iz*zWTcDsexZ76rBUoWasw9yz9zcVpPU<ka8EKY4H7y0SwM!e$cNFQ&%S6hZ%w|HQ)n^Ox1JnZP<n3f~6P>Qxn?zN|H22)R@pgk@rJ4ek+{8;tEqX~Kc5k*m8MUJ%GP92sAc$B^W!EH5u|#DC-O|#fsSy<y^suw>T)}t;__UYdqQ`TW1!2iB@4Hj$nk4{>gVWQ-U2*pMAL~wz?Tz8dG)M(@>!-S&jkh_`eH6pcS*qTy$=f0lAZQ`eYzY+lPze;fCR3|Z=7UjN>lNmWQ9Rj_9|is}Cmn-|lj*27d0gJ|qt)#|rj$`Vj%|@9ud60TW5I#Zhc!lSSYth!CGF#&#+^f@dS6HMbRAK}s^AQwLa}V=X47LcUUSKP)?|LM4zaVFW$nZdsTthldU+z8MgKoV1VD2p?AvGh7jK~Z*!al1L!}@dP)0?LJX3OW^Ftt>abk74tWv0#UMunLU|x#5m9jTPa!03qzB~0mF$lwe_A*XYn2*b@`N!skL+=@~-`K!S$*p7Yz(@|h&wy0Cq@7Vb1%PJKIstSJQZ_?8jGDZ=+hBROWuEvj7c7`kKqdw(NmcF3vo0;)`Bitrust*j_;`g}oQ|6GFR$KRdm$A#z%ut+Wu-`F-gc9hm#qzSa2Pc%1gM27#^*NaUg~k!9yH<vSb5(q3JM0a0AEsc^~_6{E?=u;q-~57fIgKB_2?fw8`Ur_jXafBt}2#XmT*ZMHf}oCsVE)FW4~yJ)~G5wol8e_^1)cqNOzSE0>*i$Bk|PZQ*~;4(c*DeTB(Jg26X@O2zGcf7}R~|;suTqwZ05O4QQPrzT>l!6nyY=bO0=AZ>-5Rb4e`&x|*?hi*G##tr!os2ofvZKy#sTyb?^jRR2)-q0g)VSb)z%*o`NfhX7v5wRNj~h^=e$sH&WUdlBswp^0hhI=Df#1>M?aeQ2|m*`bu(0~AJgQ$bZ7B<Tjlo!k!}YhU&b&)i3wc8h9cf!T_wYO`$-zz0Kkb2sffS_z~Mq9h$t$cq*oXxS()hr7KXTBnrhq#2Qh9zeUNilWQIssGrUf`!AvNY*$XZ0i<S2iR30HE3LmO$!X?hGmnh1*aezZ;QrHirmBxL}evu{>i7tpB-{aZPmm4<;=s0>*=uR2s1Xb@$4=ZsJ!5CaxQJ%7C7P*2_c8HM66s$d<OVm7t3MLAll>De*4|;^Rw7l;Dlx7+Wm)}P4h*Y@Uc6D8{5C?3FDDq;3tBCH;GbWq4nr^8I@Bl_~<Zd))U;Tx%M*L6}O7(nb7ZcZwfz*Y}_ZGs*m=!u`#$AmmfV+WRlwJk@DAT+T`#VX!NnTIC@qxj{-R=%a_KU>{A8|Fa#Z$B7eG$+-v{1u`|rQ)E`ox9nxj`bgOqwqS7t5`5~X|nDj%Hma&VuQ|!Dm&u4cW{KJE@5w((!prlYf3p*p5S)kqB^&m73l?h{xI-?}+=_4nL=Ge@7rAHc&rb$_1$dOdGG$lznlF^6+OG9$g4dmPw&SYy1-Xr#m2E&OqTxjg|LAmyx<-;E<!Ahq#JaCN}Q_!#<j((v`Fc?~HRh_t8zdjq+Q$f7Ww6NsaN^86uBYoPqb$SZ!stU12#~YY~shxJ(7RK2sNlGl?)Z}|s?)Ry1!A6UVpot}m!7mHAJMH}Z7H(_#1Oc<$&ReBt=z`O}20vc@8IJp4zCwr73glfnG1}5BoJdX+8Mw3zw?bhhHfpXV<**+OzI|$ZkicqoHAHmT&r54pyK8};lxUbGvOvYNE1Z+k*>)7!46+GAq#X3>aqqw_({_M_v9{MRT~p;craYWUi@fAFnxUh~8T+;fwyH--C^-!Vbxe;0f{+sEfW{4<2xhw9Y;AOami#6HI-PaJ${zPTm@{2FIIOl|#O6f+ld26=5!GpeRh!KwW9`rXX{Nxlx!%D0reb&8fILAVDzrYnW)QFz&E@M+6kB_6_NA>)K(cFKL~-1r=rR>Raj~>F7#ctIhvbbT?JcEKn`KhENmi?kz$8Y%Cl2BB!Db+lO2S62-131qxB~$-l^?L1Q(4UTSDO3Xrjr>Lr;4PJ+D0)^6%t+l^5jn+<JX@*e*gaSM?D%(LBe_d?F%~E{_-`jHvjVG7}9yU4E}bB=n$nG-r|O8`ffioi<ITvz4U3JEYZXMc>^4Jzgs3uB+z96U1WKj1wO;D6o%e6iMPOjWO5f|le;xg6xnXyB;!U)j(}Wuj~(*-3^7}(0}L8Ft(#O%DpFN7h!lxozccD{U<B5$7eI&Hk_`gwKK4*z1%OX-&*BwFJ)TP=Pj#IJ6#ww^q2_3Y0OoR8#C?-iOVk_i;F6VlCdw>CcLcO?`&QeoV?C=i*27%m+|B*o*91SS;(;PPbH1#QuV-al(cbLby+UsrJ1NZ1=F`a~`zAlvlHFnDB%M|nXpKGg<GLz6ROye=h9NFRvR#>TaJY7EDk!a0Gy=GYpv09$amh>L5ALpjXFcPi(?hok%+jFQkO$57O`<T_IXwPC^WW7mO;3kxW=`wgJ}j8eM3Q+g(-Ujl*m;og3h%^#(pw~D6T1gklXq-6H1baL#MyG)t|~on-=rfGrzdAP*ms=)b^}gblA!@0mHo9-yV^bhd6H=Z!KRNsDaA;<8zsJZcA6cpUEMF%$aPA$(Y2Qqu4gypHE`DEMn^1)CzmRmsk2i3D?C+_KDtKA#_4YwiE4AC3u&t^9K{%uW}c@=<8_8A<7|NMi9XVPJTA(Eh3TG5cPzAvr7Zbg-K8_9p3X#+%TxFU=W6O$Pn(WR*U0T*Y`Izl-D&f-GY7W7(P2umW+K$3u|u6?RcKyL8Dl(>qQ5@vdJ+Ol8IvNxp+ipN*7$ub6_7-+I1#6q#E_^NffeMJ|3LIRKb?p!bpsi3a=a6Hs))+Mid<rSKUZ^$ujg|i+uLRcnZib4si!y_GwXKOjx5OWK2v}YCDeDPl2*5{pR?vOh&7O70|?2+JydFK-J5d@(bAn+QqbztsN5g6y@`#$R#4o2z#t-w{z@+je0mWy;sA^y{uoE-Vdv++`b!<+g-I~Sl8<Bz+3Tq9Yiozw4?ZQ4$hAv<f5<2!$@f>+^WfjmMAY6*x6QM--1D@_{{{r|yc!$J@pZI8oK0{jrp}V)#$oP^05m*5Bdzq(nfZ-;Iou)V2lR8ORV(V3br&40m#uxZmYNiD!kcT@Mo*%*6aJ|qa~4<8akB;<w0$c)=%`?9A?`I*0HOQ|^_psrpwP5b?rbj$U}o7H6bj*D36zP!yMJl)ZjmNIG}y9e39UP%OUvz#d%vN1u3(}^V5jkCmk4O@C^kh7cND-A!cGyYY($DY#Wm;o98FeujQemIR6u}(gTS$uHDsUnO*Xg+_HgA8th`}~Kuh`&SZ2q9W0s6CK!v#xS_-758{|}~%fSGLd@GrfnC;;tx5af2G^yjBj(|cPc$p@u&CTa1PQppy&*YSfjsV;klDlP#apj=bNu`zvVZ+EzP$Fc0iS1{7W3aR7R7nmilaxdvGr3+ehU=-8ua!Zftj0VG4g#O9T*P9YVsV{^vG2Ij!dB{Z%YkGt`jf5A$m*A!RqFn^qYzDUgUJ+dM@oPVmLiG)e?Ggjnw)PyHONKtY5XKIMK%O0;^+!1)sbL77!x{hBJW9F;AUhdS#4IS1)#Za-F)Ll7OSAelzLe19Y995(MOpr4mJYMZa?Qm70rs2$=w!7FC~GJvB?`pxn^hs?bU@?s={hfe%S5EkhO86Y9y*gJ^njGs*DfzHJOQ`UOP3|IjJd?5&b}Q-x8bim0icC7Hk=Oi;MCNFt~9z+2j|5$(SUpZip>1-Y6!?z_xs*WPMpEvWRF_G_ir=U>eC9Wr<npGUgCLrk_WVx17X37K_f(cy8&eT_Bxhsv*Jxo0neiYMF(i5)}xFcbDn7G;r^mT?J8TITb8OpEb3Dfg&HA;8Kh=$aLY8sOY7^3esdwlK`2M7_q4h1qQbtwDjv{c-r{6*0ewJM?WiY>7pjN*It#1dBNUG2Tp~xPD431fSfC#nc&hW^JQtdu;4i?)JTSApa}tv#k5*3sZ<C~6DO4sM3WP9KNLjc`A)RzD0-#HlFW-yCci?v94z^S7om)TaNfJwFyplYdpm^1ilbR&U|pa9EQ+yvX~(>xO1S{xX~o;yfm5I8fi&6qeRpQYMb`_g5E+orfVnz<cGs4xlvkAXl$U{Z^E?SpNvhqBGK=+0MBVM=Z(2>TlxjE@vscd~>5D4|up~;kk!9xBDfq#W^m$>GjH8sfg6P+-l<UTG!Zr3q`a0x2+b+x7TL9Gg&EJwN@NrzI7&Y@h$8~_9u}dZjoShQ=$+=WVx8aghHIDlQ6}bhzMcr@fFARz2OE(m)7V)z4nXtv>%MHGsw4(|nYVJbrWw8Km%5sg@*Ldi_`)-fDnhm6uqT{sm)zlfsqnk9J2vfS@3=(z5e)7w>80&<I8IR+sFMXbSkV+6E%q_4{XeLtLp~*;q=@3$8SeL~V)0!FMwC!L{-i;ZyO$wk)yXp)j2I?tHgDt#dmHsUFMM|R0MIEn~!?bh7WR!IABNBrhYKqeHBFJPe_G^gBFgFKWZJcT0=V**gQmRD!JxoHNc<MIciY2|eMdPHdbq4s}jsgweG5-lh;VSu|{9KqD1+Q)9$Aa5%?i=L`)4pSlZL-j;DO|?TV$1M&3)(aZPT8cf0^hAimw;8wDv)Nq$ip#ZJ-v3ZQ07<+Yo#77JzO-AVS6N-zsZ5kw}wJ=)HNi^GZ3th;v==AuE~{kb`$QGz!j}};<8fpc-VFBC_(fhio+~*pDGnzXg(Umy6vTXf#1}n)+8c*CgYCXJ8@Cc)i*8><S#2znZu`&1GByeDN33`{YHGJHnBFz3WQ)VMRqK8sUgioG#;kVv#A#=JG2$4?u3IR3o+MXqP&$9USep?L7EJds1!9)vZOigsJcMdHLid3Ri&iK!xU1wsstw1SxCr*3xUF)#adA`$VTu>(00l86?O&`9TNrZE#>(njB(r*QmDYh_g3W+<~;@LbNA`Eca}~UnVN~Z0Y(zCe`{SC{guT0sM-W&X-qwAbFe!m`!o`ct}od=ub6i#ME)-ma^zU?@!%}E=p<TYZHZ0J<~)m-ACLZ;vLPOFR8g);oHx+mlL{ZJN*1^mS1hQY9%ml1`9e8u`pzbJK6<f!gj&^Wz}+f5T|2TZtfDqFa(P&?#k2JjJy932el&SmpN54+w9U27&?+!QsX2Pum29MhKI;oK*zpl-TQ^y@bOogssRj=ol-vTz%3kvnu5Hq4#rXsmd{f>ruu&gY!`Y!!Y3ixlx5-Fc%L04Qazb75gaMkH(Y2vY2V3&hj_Sz$Z;Hmq8Oy@fDK8P)_d4g7vE<H1VHW2gu_&&O5fU5SIi2h(qQ#nif~ggkx>mK@(ynFSOnLXG4fpT$irOy0v8K(|C%hssiRplI1k2M$4~3mdJ<Tzz;oASf&t{@xSC%@2;Jy{L(5EJCmo}ccEkyr!4NljsxqDx|z_)BjCb_O}m82{tz}B=GE3(XVnREhNTsr;yT=4^@wtR7WyYrIv%5Uc>E-7ykjQYX80a<1GqKdQ#qvyF~)8^=_<O=teY5Mvy(k82e@$0Tj=eC!nUw?*3PHR3&1?=c$#j!yIit9V<&9L40vqmek7Zik<PYpe3zc<?&sVHt7ujT;s1TsNhzxB$Q2d}U1q^=qS6+!TAidp3L;MU0h=F}myyMX=;7vPq)tJv)MQkg;*jdCb~VU{jW=B|v_W*e9LIo}ax^~Q)F^-wo&Lp>F)n(GTAmeQ4$vq%%+R*Ig(degR8Xjz~D4|@aCr#Ro1{*B3132Nx9_p)*m?br;NY-1>APi-NE>Wm_y)Al6g#}Dd?jT@$mD($o2?9Df9{1T~ouKbfMb9<JxUF@TI*e*6r6-iB8Nfq<RAt8^+79wRVra@-8=Yc$Vf^dwK)lYNote}tQ>PSt)u9s#mbgs-wk!8av>&Je*q{nF$GT1bXZtRHz4t{aLN;c%Q23kf%i>6`fQ2zq<n^`N~D_7_hif6q#)GNi)a#WrKbZZ?i@5ixT=C`*N8=tNem2vYzr$%RvNCt?|BGb&E4K@|JuRuv73TI0O@Ng8`D9=Xqa!Z?ZY82vt*=Z_t0Whs{#nfmG({6Crd9B`oT<Q=rv2Y&Qx7#vOIeJt-+Y4BLRvIu5SU&qs;XbBrMj<!WcUw>N?vh@*G5Y03O^sWgs6!ALf1ONSGGIQQ<~!Je*yT}OA95eq&aY|FnKTBvFlqXysb=*kS3B(rQ3KD2GC@BxI&ftrh?zLqGDNIgrt*lX^q&X*J(k#;q!69Dz-qgND{ADz-}W(3RHJh!Y&sRy@%qKdR;E@kPjNrk+y;hsLHgz#P}>gK8A5qa<=&p~Z;BxXDVs-6v}oMXHACyImNGFzH})O%pYZ)GLp|)^@dCvQrfcln%WOmXbu*G=+vguYmsoJcEnSl6pBmXdgp&s$LMmG6Xv;DZcQlIklEmNvYWH17YdYpCt$n3`r~R?OQ(HIVvO(J#5k3fHwrJ2^x%}-POzD@Tf*iZjUZA)Pso8_f<@TGg1FL&%Hvv^fB{k3t1g9VI61VhbVC2$_MhiQ%g>k2w3Cl$XGB2JhZHsz`XODMtQau}7A-71Fu_I}L$`1U8ek##Ct7d8h13{{@)*#>tDn@*m<zji-JH5kjn^>PTA@o>7h;=WqL%3pzygu#7SQo|u7b+-s{j8Jg_>Q|YaB}6Yy!uwfSZ2ZZzO?tShPqWe4ocf1%fES8au4xXpy5XjfMcC56lXr1S&!o0MNL1lqch%0y9rJmo1nA!d*h03l>&~9r%xMqqtH@i(#^7nqU|djM%N_c7`_`sYss1JLY7uNO@psZAx`ZG1Hv;sjRW|r!{8GDC^ATWLfDUS8Er~IzC9bkK%&hA4ttwEr$SwX%MX^wXp`VsKA5TkV(IXM)=38(9C#vCfDzrn@@40O#tIwRK6F%YtKgh_37nYbjIN&oXiMOiYS+lmpv@;gh8%QrSxBBbp4J^cPrTf0qQm_1?Q1`z44-r2yHno-tkrt6?Nhh03LMJZOGgf@)*UW@KS`0WhlJI)fT|8ML64=x=O7BFi@7}3;SnmjUGF1>Xajq>w|JsQV9F2d!`>d+>!}h>7b*l)Qs<u>DgLKgO8B*-m9zxLFO|ZjM76wwua)#_ckR|>Cy)t?_ujVO{q)_3Z~yeo$4mX$IL}aN%;R&>O~9t_&@+83eX*mo6Q{wK`u^bM7a}{^h=pBUAVGj-0r>V&C`c${7b{gvhf|GL4XjyTTbQNKHveX%F>t84Q68MbJjcWaqJC<Z21O1jEa)NMvqxFnoSJo9{uX&%C_Suiqh30oq6EB*T1VXxrHr5HAWe;YA+xWStgQ!N;bj%i(nR8f!I4I(C2;xdJL<y7pc7R@btN>x_>VTR6rW-=jmfGe#1#DQq_f#KX}z-PD3J)kOH?^cBn7l3$$-&5ba<13fdPx4aQ<Az+^xt9;#Ns10>W4KK%GlVPnBur#plzsIze2`Cb`iG1Z`tat%%!Gs=s?jUK;J+d4j8m`EpFU$)##vS~^tf_JKO}^LWVxV*ur0W1C5x1klc1MJ$FC(Yct@mxk&pPo&xAO*bE!xmABhUru{lxeHP!AE-GlZZ!MTq}ycSoBbLK)dKEiU_V{lPDfe)_G9=cKV`w|kDtdTi!OE-X1p0sk5@OV*4UkmcxWZnSTxN@Ob})0**zEzg~MiBk@u)ejf;E2qTV*bgqMa76`@RFvWVs?Dpe9EIW&4`w0b$`vcvaMF<v2<a%rGcK%-Gk;i&;DEVZpb+s0nu89YX?v7m{wfbgdobZ69nKWzH!{qjNpf9VlM9`j5Nn?r|bNSzdJ4p!#>*k=Ip)ydYRK|ms<tecdV!{$^m3zhmyk|xKDR~A_%OXJ%9?WSZdTB)6AtqSX%j5vpc56iN34%@9=?~Hc7IYaQk+<xSoI58E4%9PSh4of=8JZ1V-2mb%k#oe{p^NaN|XfW1x9Z#>-Of2=BqZQ#0IC0XuvHFxR!#YMR;%M>r2!GPPPu0G^ezRwzp*$c3^W1tHhbC^l#&eU@Dmoh5Z5)pA+O0`+OQmaq*<_)`+s_eY&~TM%EBK(i@qBSYubY>WZEp-;t^{=A+J&RL3Bi?o`+)6>TRbpdemX^S=4a3Pln_-A!D9C9=a1jN|NOE2wjsehu7l?fAAe}Sd^xL|COoc=FI^hG_NGmpijfJL=#$bPlCKN_?7`3f=}4e@{ol&DQro`D^6Y|h(l6d7o#>#d%SG8?VA@B%dfbext~^cDpQE36?=2<0`QpRMEyu2rRp)bsjxMCbJsvybn{ndm_moYP_+_Xumwe3Ys5hY~9&Behs0?Lfo+ktHAJ&Mqs3Lg0D44|+WFh!n3uUd)!R=}VADhU=_04WK7f2TM_rAguq3GGQ^XTosB&Ng-Bgbb+p9^I(xD4C1&sPI%lWbas62JKd-upy+6{cfm_j*CP<y{^tP~fi{!CTtd7B?i~qYTx~SUJ#RvViZ@<$JLFGdOKV>1mNcf<Dob%d#X1cZUfuniZt)gAC*N+yMay$MFM<(%2(F!RQa|_AjjH2k$p=e~JH#9bn`ECp+bdN6f=^gP34-m}z?ZZ^N8{jUw<7PovEJx*z8a=QB`#vK!z(`@qW-400m^UX^icIvHU9i3a#U81%1?K>AN)urVd<w~-GcuaIQNvJ*tX-v;}FE$?2i<u$>lS9y)U-Hc3A7_6L~31rx!H5jXZ21Z0XkHZEJ$j^WM>;DG@5Jf)')))
_ROUTES={0:_PAYLOAD['base']}
for _rid,_patch in _PAYLOAD['patches'].items():
    _tape=list(_ROUTES[0])
    for _t,_a in _patch: _tape[_t]=_a
    _ROUTES[int(_rid)]=_tape
del _PAYLOAD
_SETTINGS={'hand_align': True, 'weed_repair': True, 'sell_lead': True, 'budget_guard': False, 'room_guard': False, 'clamp_sells': False, 'dead_stock': False, 'terminal_liquidation': False, 'front_run': False}

def _router(observation,step,state):
    if step>=144 and not state.get('day6'):
        shops=_get(_get(observation,'town',{}),'unlocked_shops',[]) or []
        state['route']=1 if 'YARN_STORE' in shops else 0
        state['day6']=True
    if step>=648 and not state.get('day27'):
        inv=_get(_get(observation,'market',{}),'inventory',{}) or {}
        state['route']=2 if _int(_get(inv,'EGG',10000))<=9888 else 3
        state['day27']=True
    return state.get('route',0)

_IMPL=make_agent(_ROUTES,router=_router,**_SETTINGS)
_IMPL.chassis.diagnostics['terminal_rescue_errors']=0

def agent(observation,configuration=None):
    try:
        action=_IMPL(observation,configuration)
        pass
        return action
    except Exception:
        return {'farmer':['PASS'],'hands':[],'market':[]}

_SHOP_PARENT=agent
del agent

def agent(observation,configuration=None):
    action=_SHOP_PARENT(observation,configuration)
    try:
        if _step_of(observation)>=718:
            view=_View(observation,_int(_get(observation,'player',0)),_IMPL.chassis.cfg)
            units=[]
            for i,pos in enumerate(view.positions):
                units.append(['DROP'] if _shed_adjacent(pos,view.board) and view.inv(i) else ['PASS'])
            action={'farmer':units[0],'hands':units[1:],'market':[]}
            projected=_IMPL.chassis._projected_shed(action,view)
            action['market']=[['SELL',item,projected.get(item,0)] for item in PRODUCTS if projected.get(item,0)>0]
            action['market'].sort(key=lambda o:-view.prices.get(o[1],0)*o[2])
    except Exception:
        _IMPL.chassis.diagnostics['terminal_rescue_errors'] += 1
    return action


## 3. Verify source integrity and actual Kaggle loading


In [ ]:
import ast, contextlib, hashlib, io, json

MAIN = WORKDIR / 'main.py'
EXPECTED_SHA256 = 'bb3109331d4087d5e5890515f9b1d1e0d4f9ea6015ee3d1b172a93b8b1d8dbb4'
source_bytes = MAIN.read_bytes().replace(b'\r\n', b'\n')
MAIN.write_bytes(source_bytes)
assert hashlib.sha256(source_bytes).hexdigest() == EXPECTED_SHA256, 'Source integrity check failed.'
compile(source_bytes, 'main.py', 'exec')
tree = ast.parse(source_bytes)
imports = [n.module.split('.')[0] for n in ast.walk(tree) if isinstance(n, ast.ImportFrom)]
imports += [a.name.split('.')[0] for n in ast.walk(tree) if isinstance(n, ast.Import) for a in n.names]
assert all(name in sys.stdlib_module_names for name in imports), imports
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    from kaggle_environments import make
    from kaggle_environments.agent import get_last_callable
entry = get_last_callable(source_bytes.decode('utf-8'), path=str(MAIN))
assert entry.__name__ == 'agent', 'Unexpected final callable.'

games = []
for label, opponent in [('self-play', str(MAIN)), ('starter', 'starter')]:
    for seed in (7, 1234):
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            env = make('kaggriculture', configuration={'episodeSteps': 720, 'seed': seed})
            final = env.run([str(MAIN), opponent])[-1]
        statuses = [row['status'] for row in final]
        rewards = [row['reward'] for row in final]
        assert statuses == ['DONE', 'DONE'], (label, seed, statuses)
        games.append({'opponent': label, 'seed': seed, 'rewards': rewards, 'status': statuses})
        print(label, seed, statuses, rewards)
assert all(row['rewards'][0] > row['rewards'][1] for row in games if row['opponent'] == 'starter'), 'Starter smoke failed.'
print('Source, entry point, self-play and starter checks passed.')


## 4. Create the submission archive

Download the archive from this cell or the notebook Output files. Upload it through Kaggle only when you choose to submit.


In [ ]:
import gzip, io, tarfile
from IPython.display import FileLink, display

ARCHIVE = WORKDIR / 'submission_competitive_v25.tar.gz'
with ARCHIVE.open('wb') as file:
    with gzip.GzipFile(filename='', mode='wb', fileobj=file, mtime=0) as compressed:
        with tarfile.open(fileobj=compressed, mode='w', format=tarfile.GNU_FORMAT) as tar:
            info = tarfile.TarInfo('main.py')
            info.size, info.mode, info.mtime = len(source_bytes), 0o644, 0
            tar.addfile(info, io.BytesIO(source_bytes))
with tarfile.open(ARCHIVE) as tar:
    assert tar.getnames() == ['main.py']
    assert tar.extractfile('main.py').read() == source_bytes
manifest = {'main_sha256': EXPECTED_SHA256,
            'archive_sha256': hashlib.sha256(ARCHIVE.read_bytes()).hexdigest(),
            'games': games, 'validation': 'PASS'}
(WORKDIR / 'v25_manifest.json').write_text(json.dumps(manifest, indent=2))
print('VALIDATION PASS:', ARCHIVE.name, ARCHIVE.stat().st_size, 'bytes')
print('SHA-256:', manifest['archive_sha256'])
display(FileLink(ARCHIVE.name))


# Attribution

- [yhay81 — Shop Router 0908](https://www.kaggle.com/code/yhay81/shop-router-0908):
  the four public production schedules and routing decisions.
- yhay81's Fieldbook sale-leading ideas; thomastschinkel and tetsutani:
  previously audited runtime/routing, weed-handling and inventory work.
  The inherited runtime's Apache-2.0 license is embedded in `main.py`.
- [mohui666 — v66](https://www.kaggle.com/code/mohui666/kaggriculture-v66-public-agent)
  and TITAN: terminal cargo/deadline research informed independent experiments.
  No donor terminal implementation or hidden-state signature is embedded.
- destbreso and nikital7: prior simulator and measurement research; those tools
  are not shipped in the agent.
- All other reviewed sources and rejected ideas are listed in `SOURCE_AUDIT.md`.

The new integration, packing, ablations and validation are EXP-149 work for
Ahmed Berat Özer. Public source credit is retained; no claim of authorship is
made over donor schedules. The final agent has no v24 production payload.
